# Géomarketing IDF : J13c — Arrêts IDFM et équipements BPE dans les IRIS d’Île-de-France

## Objectifs
- Partir directement des points d’arrêt IDFM et du fichier détail BPE.
- Extraire une seule fois les observations BPE situées en Île-de-France.
- Préparer les restaurants A504 et les autres équipements dans deux tables distinctes.
- Conserver les identifiants, noms diffusés, catégories et coordonnées d’origine.
- Affecter les points aux contours IRIS complets de J13b et diagnostiquer les exclusions.
- Calculer des nombres, densités et parts et conserver les catégories en format long.
- Cartographier l’offre et les transports sans API key ni fond OpenStreetMap.
- Documenter les sources et enregistrer le notebook sur GitHub sans sorties volumineuses.
## Périmètre
Tous les IRIS d’Île-de-France conservés dans J13b, avec un focus sur Cergy et Pontoise.
Le profil reprend RP 2022 / géographie IRIS 2024. La BPE 2025 est diffusée en géographie
2025 : son millésime et sa géographie restent distincts de ceux du profil. Les points
sont affectés par leur position aux polygones de l’étude ; une commune contradictoire
reste dans le diagnostic. Les valeurs et statuts fiscaux de J13b sont conservés.
## Interprétation
Les comptes décrivent les observations retenues après contrôle, pas un inventaire
garanti exhaustif des établissements réels. Les transports sont comptés selon
l’identifiant et l’unité déclarés. Une zone d’arrêt, un quai et une gare commerciale
ne sont pas interchangeables. Les trajets, fréquences, voyageurs et temps de marche
ne sont pas calculés à partir de ces seuls points. Les mobilités seront traitées au J13d.

In [41]:
#Imports et environnement

import os
os.environ['OMP_NUM_THREADS']='1'
os.environ['NUMEXPR_NUM_THREADS']='1'
os.environ['MKL_NUM_THREADS']='1'

from pathlib import Path
from datetime import datetime, timezone
from html import escape
import csv
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import geopandas as gpd
import plotly
import plotly.graph_objects as go
from IPython.display import display

pd.set_option('display.max_columns',80)
pd.set_option('display.max_rows',20)
print('Pandas :',pd.__version__,'; GeoPandas :',gpd.__version__,'; Plotly :',plotly.__version__)


Pandas : 2.2.2 ; GeoPandas : 1.0.1 ; Plotly : 5.24.1


In [42]:
#Chemin et inventaire des entrées disponibles

RACINE=Path(os.environ.get('GEOMARKETING_RACINE',r'C:\Users\almou\OneDrive\GeoMarketing_IDF'))
FICHIER_PROFIL_J13B=RACINE/'data/processed/j13b/profil_iris_idf_j13b.csv'
FICHIER_GEO_J13B=RACINE/'data/processed/j13b/profil_iris_idf_j13b.gpkg'
COUCHE_IRIS='profils_iris'
DOSSIER_INTERIM=RACINE/'data/interim/j13c'
DOSSIER_PROCESSED=RACINE/'data/processed/j13c'
DOSSIER_ENTREES=DOSSIER_PROCESSED/'entrees'
DOSSIER_CARTES=RACINE/'outputs/cartes/j13c'
DOSSIER_DOCS=RACINE/'docs'
for dossier in [DOSSIER_INTERIM,DOSSIER_PROCESSED,DOSSIER_ENTREES,DOSSIER_CARTES,DOSSIER_DOCS]:
    dossier.mkdir(parents=True,exist_ok=True)

inventaire=[]
for zone in ['raw','interim','processed']:
    for fichier in sorted((RACINE/'data'/zone).rglob('*')):
        if fichier.is_file() and fichier.suffix.lower() in {'.csv','.xlsx','.geojson','.json','.gpkg','.zip'}:
            if DOSSIER_PROCESSED in fichier.parents or DOSSIER_INTERIM in fichier.parents:continue
            inventaire.append({'ZONE':zone,'CHEMIN':str(fichier.relative_to(RACINE)),
                'FORMAT':fichier.suffix.lower(),'TAILLE_MO':round(fichier.stat().st_size/1_000_000,2)})
inventaire=pd.DataFrame(inventaire,columns=['ZONE','CHEMIN','FORMAT','TAILLE_MO'])
inventaire.to_csv(DOSSIER_INTERIM/'inventaire_entrees_j13c.csv',index=False,encoding='utf-8-sig')
print('Racine :',RACINE)
print('Profil J13b présent :',FICHIER_PROFIL_J13B.is_file())
print('Contours J13b présents :',FICHIER_GEO_J13B.is_file())
display(inventaire.head(80))
display(inventaire[inventaire['CHEMIN'].str.contains(r'idfm|arret|arrêt|gare|stop|transport|bpe|equip',case=False,na=False)])


Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
Profil J13b présent : True
Contours J13b présents : True


,ZONE,CHEMIN,FORMAT,TAILLE_MO
0,raw,data\raw\bpe\BPE25.csv,.csv,1515.25
1,raw,data\raw\bpe\DS_BPE_2025_data.csv,.csv,149.89
2,raw,data\raw\geographie\communes_idf_api.geojson,.geojson,6.86
3,raw,data\raw\geographie\communes_idf_api_metadata....,.json,0.00
4,raw,data\raw\geographie\iris\iris_2024.geojson,.geojson,11.65
...,...,...,...,...
75,interim,data\interim\j10e\sports_loisirs_culture_commu...,.csv,0.48
76,interim,data\interim\j10f\bilan_categories.csv,.csv,0.00
77,interim,data\interim\j10f\controle_coordonnees.csv,.csv,0.00
78,interim,data\interim\j10f\controle_effectifs_source.csv,.csv,0.00


,ZONE,CHEMIN,FORMAT,TAILLE_MO
0,raw,data\raw\bpe\BPE25.csv,.csv,1515.25
1,raw,data\raw\bpe\DS_BPE_2025_data.csv,.csv,149.89
5,raw,data\raw\idfm\arrets.geojson,.geojson,19.05
6,raw,data\raw\idfm\traces-des-lignes-regulieres-de-...,.geojson,104.25
7,raw,data\raw\idfm\traces-du-reseau-ferre-idf.geojson,.geojson,3.23
71,interim,data\interim\j10e\equipements_hors_profil.csv,.csv,0.00
125,interim,data\interim\j11b\arrets_coordonnees_a_verifie...,.csv,0.00
126,interim,data\interim\j11b\arrets_geometries_a_verifier...,.csv,0.00
127,interim,data\interim\j11b\inventaire_lignes_idfm.csv,.csv,0.07
134,interim,data\interim\j11b\synthese_modes_idfm.csv,.csv,0.00


In [53]:
# Configurer les 3 sources de points

ANNEE_RP=2022
GEOGRAPHIE_IRIS=2024
DEPARTEMENTS_IDF={'75','77','78','91','92','93','94','95'}
CRS_TRAVAIL=2154
AFFICHER_CARTES_DANS_NOTEBOOK=False
SIMPLIFICATION_CARTE_METRES=10
CODES_FOCUS=['95127','95500']  # Vide : reprendre J13b. Exemple : ['95127','95500'].
FICHIER_CORRECTIONS_LIMITES=None  # CSV validé, créé à partir du modèle du bloc 16.
CONFIG_PLOTLY={'scrollZoom':True,'displaylogo':False}


# Chemins réels à renseigner après l'inventaire. Adapter aussi SOURCES ci-dessous.
FICHIER_RESTAURANTS_EXISTANT=None  # None : restaurants construits depuis la BPE.
FICHIER_ARRETS_IDFM=RACINE/"data/raw/idfm/arrets.geojson"
FICHIER_EQUIPEMENTS_DETAIL=None  # None : équipements construits depuis la BPE.
FICHIER_BPE_DETAIL=RACINE/'data/raw/bpe/BPE25.csv'  # Adapter au vrai fichier CSV ou ZIP.
ANNEE_BPE=2025
GEOGRAPHIE_BPE=2025
DATE_SOURCE_BPE=None
ANNEE_SOURCE_IDFM=None
DATE_SOURCE_IDFM=None
URL_NOTICE_BPE='https://www.insee.fr/fr/statistiques/8217525'
URL_NOTICE_IDFM=None  # Lien du jeu réellement utilisé, à compléter pour la provenance.
COLONNES_CLE_TRANSPORTS=[]  # Vide : utiliser un seul identifiant déclaré/reconnu.
UNITE_TRANSPORTS='Point de transport selon l’identifiant et la granularité du fichier IDFM sélectionné'
MEMBRE_CSV_BPE=None  # Si le ZIP contient plusieurs CSV compatibles, préciser le membre.
TAILLE_BLOC_BPE=50_000
FICHIER_NOMENCLATURE_BPE=None
COLONNE_CODE_NOMENCLATURE=None
COLONNE_LIBELLE_NOMENCLATURE=None

# Les noms suivants seront créés par les blocs 8–10.
FICHIER_BPE_IDF=DOSSIER_INTERIM/'bpe_detail_idf_j13c.csv'
FICHIER_ARRETS_PREPARES=DOSSIER_ENTREES/'arrets_idfm_idf_j13c.csv'
FICHIER_EQUIPEMENTS_PREPARES=DOSSIER_ENTREES/'equipements_idf_j13c.csv'
FICHIER_RESTAURANTS_PREPARES=DOSSIER_ENTREES/'restaurants_idf_j13c.csv'
PREPARATIONS_EFFECTUEES=[]
NOMENCLATURE=pd.DataFrame(columns=['CODE','LIBELLE'])

SOURCES={
 'RESTAURATION':{
    'actif':True,'fichier':FICHIER_RESTAURANTS_EXISTANT,
    'couche':None,'feuille':0,'id':None,'nom':None,'categorie':None,
    'x':None,'y':None,'coordonnees':None,'ordre_coordonnees':'LAT_LON','crs':4326,
    'commune':None,'annee_source':ANNEE_BPE,'date_source':DATE_SOURCE_BPE,
    'geographie_source':GEOGRAPHIE_BPE,'url_notice':URL_NOTICE_BPE,
    'unite':'Observation de restaurant BPE, dédupliquée selon l’identifiant choisi',
    'autoriser_id_ligne':True,
 },
 'TRANSPORTS':{
    'actif':True,'fichier':FICHIER_ARRETS_IDFM,
    'couche':None,'feuille':0,'id':None,'nom':None,'categorie':None,
    'x':None,'y':None,'coordonnees':None,'ordre_coordonnees':'LAT_LON','crs':4326,
    'commune':None,'annee_source':ANNEE_SOURCE_IDFM,'date_source':DATE_SOURCE_IDFM,
    'url_notice':URL_NOTICE_IDFM,
    'unite':UNITE_TRANSPORTS,
    'autoriser_id_ligne':False,
 },
 'EQUIPEMENTS':{
    'actif':True,'fichier':FICHIER_EQUIPEMENTS_DETAIL,
    'couche':None,'feuille':0,'id':None,'nom':None,'categorie':None,
    'x':None,'y':None,'coordonnees':None,'ordre_coordonnees':'LAT_LON','crs':4326,
    'commune':None,'annee_source':ANNEE_BPE,'date_source':DATE_SOURCE_BPE,
    'geographie_source':GEOGRAPHIE_BPE,'url_notice':URL_NOTICE_BPE,
    'unite':'Observation d’équipement complémentaire de la table de détail',
    'autoriser_id_ligne':True,
 },
}
if not any(c['actif'] for c in SOURCES.values()):
    raise ValueError('Activer au moins une source de points.')


In [54]:
#Fonctions pour les noms, les codes
def normaliser_nom(nom):
    s=unicodedata.normalize('NFKD',str(nom).strip().upper())
    s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^A-Z0-9]+','_',s).strip('_')

def entetes(table):
    geom=table.geometry.name if isinstance(table,gpd.GeoDataFrame) else None
    noms=[c if c==geom else normaliser_nom(c) for c in table.columns]
    if len(noms)!=len(set(noms)):
        raise ValueError('En-têtes identiques après normalisation.')
    table=table.copy();table.columns=noms
    return table

def nombre(serie):
    texte=serie.astype('string').str.strip().str.replace(r'[\s\u00a0\u202f]','',regex=True)
    return pd.to_numeric(texte.str.replace(',','.',regex=False),errors='coerce').replace([np.inf,-np.inf],np.nan)

def texte(serie):
    s=serie.astype('string').str.strip()
    return s.mask(s.str.upper().isin(['','NAN','NONE','<NA>','NULL','ND','[ND]']))

def codes(serie,longueur):
    s=texte(serie).str.upper().str.replace(r'\.0$','',regex=True)
    masque=s.str.fullmatch(r'\d{1,'+str(longueur)+'}',na=False)
    s.loc[masque]=s.loc[masque].str.zfill(longueur)
    return s

def booleen(serie,nom):
    v=texte(serie).str.upper().map({'TRUE':True,'FALSE':False,'1':True,'0':False,'VRAI':True,'FAUX':False})
    if v.isna().any():raise ValueError(nom+' : booléen non reconnu.')
    return v.astype(bool)

def exporter_csv(table,path):
    table.to_csv(path,index=False,encoding='utf-8-sig')

def empreinte(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def dialecte_csv(path):
    with Path(path).open('rb') as flux:echantillon=flux.read(32768)
    try:s=echantillon.decode('utf-8-sig');enc='utf-8-sig'
    except UnicodeDecodeError:s=echantillon.decode('cp1252');enc='cp1252'
    ligne=s.splitlines()[0] if s.splitlines() else ''
    candidats={sep:next(csv.reader([ligne],delimiter=sep)) for sep in [';',',','\t']}
    sep=max(candidats,key=lambda sep:len(candidats[sep]))
    if len(candidats[sep])<2:raise ValueError('CSV vide ou séparateur/en-tête non reconnu : '+str(path))
    return sep,enc

def apercu_source(cfg):
    p=Path(cfg['fichier']);suffixe=p.suffix.lower()
    if suffixe=='.csv':
        sep,enc=dialecte_csv(p)
        return entetes(pd.read_csv(p,sep=sep,encoding=enc,dtype='string',nrows=5))
    if suffixe=='.xlsx':
        return entetes(pd.read_excel(p,sheet_name=cfg['feuille'],engine='openpyxl',dtype='string',nrows=5))
    if suffixe in {'.gpkg','.geojson','.json'}:
        return entetes(gpd.read_file(p,**({'layer':cfg['couche']} if cfg['couche'] else {}),rows=5))
    raise ValueError('Format de points attendu : CSV, XLSX, GeoJSON ou GeoPackage.')

def choisir(champs,explicite,alias,nom,obligatoire=False,priorite=False):
    if explicite is not None:
        c=normaliser_nom(explicite)
        if c not in champs:raise ValueError(nom+' : colonne absente : '+c)
        return c
    possibles=[c for c in alias if c in champs]
    if priorite and possibles:return possibles[0]
    if len(possibles)>1:raise ValueError(nom+' : plusieurs colonnes possibles '+str(possibles)+'. Préciser le paramètre.')
    if possibles:return possibles[0]
    if obligatoire:raise ValueError(nom+' : colonne introuvable. Colonnes : '+str(list(champs)))
    return None

LECTURES=[]
STATISTIQUES_SOURCES=[]
DICTIONNAIRE=[]




In [55]:
# Inspecter les sources avant de les charger

travail=[]
for source,cfg in SOURCES.items():
    fichier=cfg['fichier']
    existe=fichier is not None and Path(fichier).is_file()
    if not cfg['actif']:
        statut='SOURCE_DESACTIVEE'
        action='Préparer cette source avant une intégration future ; indicateurs NA.'
    elif existe:
        statut='ENTREE_A_INSPECTER'
        action='Vérifier identifiant, catégorie, coordonnées et unité comptée.'
        cfg['fichier']=Path(fichier)
        ap=apercu_source(cfg)
        exporter_csv(pd.DataFrame({'COLONNE':ap.columns}),DOSSIER_INTERIM/f'entetes_{source.lower()}_j13c.csv')
        print(source,':',fichier);print(ap.columns.tolist());display(ap.head(5))
    elif source in ['RESTAURATION','EQUIPEMENTS'] and FICHIER_BPE_DETAIL is not None:
        statut='DETAIL_BPE_A_EXTRAIRE'
        action='Vérifier le détail BPE CSV/ZIP puis préparer l’extrait régional au bloc 8.'
    else:
        statut='FICHIER_A_RENSEIGNER_OU_RETROUVER'
        action=('Renseigner au bloc 3 les points d’arrêt IDFM ; conserver les tracés dans J12.'
            if source=='TRANSPORTS' else
            'Renseigner le détail des équipements ou FICHIER_BPE_DETAIL ; les totaux communaux ne suffisent pas.'
            if source=='EQUIPEMENTS' else 'Renseigner le détail BPE pour construire les restaurants, ou choisir un détail déjà préparé.')
    travail.append({'SOURCE':source,'ACTIF':cfg['actif'],'FICHIER_RENSEIGNE':str(fichier) if fichier is not None else '',
                    'STATUT':statut,'TRAVAIL_A_REPRENDRE':action})
travail=pd.DataFrame(travail)
exporter_csv(travail,DOSSIER_INTERIM/'travail_a_reprendre_j13c.csv')
display(travail)


TRANSPORTS : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\idfm\arrets.geojson
['ARRID', 'ARRVERSION', 'ARRCREATED', 'ARRCHANGED', 'ARRNAME', 'ARRTYPE', 'ARRXEPSG2154', 'ARRYEPSG2154', 'ARRTOWN', 'ARRPOSTALREGION', 'ARRACCESSIBILITY', 'ARRAUDIBLESIGNALS', 'ARRVISUALSIGNS', 'ARRFAREZONE', 'ZDAID', 'geometry']


,ARRID,ARRVERSION,ARRCREATED,ARRCHANGED,ARRNAME,ARRTYPE,ARRXEPSG2154,ARRYEPSG2154,ARRTOWN,ARRPOSTALREGION,ARRACCESSIBILITY,ARRAUDIBLESIGNALS,ARRVISUALSIGNS,ARRFAREZONE,ZDAID,geometry
0,15533,1236744-1236745,2014-12-29 01:00:00+01:00,2026-06-29 10:38:35+02:00,Côteaux,bus,679307,6865631,Thorigny-sur-Marne,77464,false,unknown,unknown,5,55894,POINT (2.71779 48.89036)
1,478686,1180049-58188,2019-11-27 01:00:00+01:00,2026-06-29 10:38:33+02:00,Cordier,bus,679265,6858294,Ferrières-en-Brie,77181,true,unknown,unknown,5,58188,POINT (2.71758 48.82437)
2,11079,1227239-52687,2014-12-29 01:00:00+01:00,2026-06-29 10:38:30+02:00,Corbie,bus,685502,6867261,Coupvray,77132,false,unknown,unknown,5,52687,POINT (2.80222 48.90519)
3,15232,1319095-55776,2014-12-29 01:00:00+01:00,2026-06-29 10:38:29+02:00,Commissariat,bus,684307,6862313,Chessy,77111,false,unknown,unknown,5,55776,POINT (2.78611 48.86066)
4,15566,15566-47267,2014-12-29 16:31:51+01:00,2026-06-29 10:38:27+02:00,Collège Victor Schoelcher,bus,674989,6859528,Torcy,77468,false,unknown,unknown,5,47267,POINT (2.65927 48.83532)


,SOURCE,ACTIF,FICHIER_RENSEIGNE,STATUT,TRAVAIL_A_REPRENDRE
0,RESTAURATION,True,,DETAIL_BPE_A_EXTRAIRE,Vérifier le détail BPE CSV/ZIP puis préparer l...
1,TRANSPORTS,True,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,ENTREE_A_INSPECTER,"Vérifier identifiant, catégorie, coordonnées e..."
2,EQUIPEMENTS,True,,DETAIL_BPE_A_EXTRAIRE,Vérifier le détail BPE CSV/ZIP puis préparer l...


In [56]:
#Charger le profil J13b

for p in [FICHIER_PROFIL_J13B,FICHIER_GEO_J13B]:
    if not p.is_file():raise FileNotFoundError('Exécuter J13b ou adapter le chemin : '+str(p))
base=entetes(pd.read_csv(FICHIER_PROFIL_J13B,sep=None,engine='python',dtype='string',encoding='utf-8-sig'))
requis={'IRIS','CODGEO_SOURCE','CODGEO_ETUDE','NOM_IRIS','NOM_COMMUNE_ETUDE',
        'J13A_POPULATION','J13A_ANNEE_RP','J13A_GEOGRAPHIE','J13B_DANS_FOCUS'}
if requis-set(base):raise ValueError('Colonnes J13b absentes : '+str(sorted(requis-set(base))))
if any(c.startswith('J13C_') for c in base):raise ValueError('L’entrée doit être J13b, pas une ancienne sortie J13c.')
base['IRIS']=codes(base['IRIS'],9)
if not base['IRIS'].str.fullmatch(r'\d{9}',na=False).all() or not base['IRIS'].is_unique:
    raise ValueError('Codes IRIS J13b invalides ou dupliqués.')
if set(base['IRIS'].str[:2])!=DEPARTEMENTS_IDF:raise ValueError('Les huit départements IDF doivent être présents.')
if not nombre(base['J13A_ANNEE_RP']).eq(ANNEE_RP).all() or not nombre(base['J13A_GEOGRAPHIE']).eq(GEOGRAPHIE_IRIS).all():
    raise ValueError('RP/géographie J13b incompatibles avec les paramètres.')
for c in ['CODGEO_SOURCE','CODGEO_ETUDE']:base[c]=codes(base[c],5)
if not base['CODGEO_SOURCE'].eq(base['IRIS'].str[:5]).all():raise ValueError('Codes commune source / IRIS incohérents.')
base['J13B_DANS_FOCUS']=booleen(base['J13B_DANS_FOCUS'],'J13B_DANS_FOCUS')
if CODES_FOCUS:
    focus_codes=set(codes(pd.Series(CODES_FOCUS),5))
    if focus_codes-set(base['CODGEO_ETUDE']):raise ValueError('Commune du focus absente de J13b.')
    base['J13C_DANS_FOCUS']=base['CODGEO_ETUDE'].isin(focus_codes)
else:
    base['J13C_DANS_FOCUS']=base['J13B_DANS_FOCUS']
    focus_codes=set(base.loc[base['J13C_DANS_FOCUS'],'CODGEO_ETUDE'])
contours=entetes(gpd.read_file(FICHIER_GEO_J13B,layer=COUCHE_IRIS))
if 'IRIS' not in contours or contours.crs is None:raise ValueError('Contours : code IRIS et CRS obligatoires.')
contours['IRIS']=codes(contours['IRIS'],9)
if not contours['IRIS'].is_unique or set(contours['IRIS'])!=set(base['IRIS']):
    raise ValueError('Clés CSV et GeoPackage J13b différentes ou dupliquées.')
if contours.geometry.isna().any() or contours.geometry.is_empty.any() or not contours.geometry.is_valid.all():
    raise ValueError('Contour vide ou invalide : corriger la géométrie en amont.')
if not contours.geometry.geom_type.isin(['Polygon','MultiPolygon']).all():raise ValueError('Des polygones IRIS sont attendus.')
iris_polygones=contours[['IRIS',contours.geometry.name]].rename_geometry('geometry') if contours.geometry.name!='geometry' else contours[['IRIS','geometry']]
iris_polygones=iris_polygones.to_crs(CRS_TRAVAIL).reset_index(drop=True)
ATTENDUS=set(base['IRIS']);NOMBRE_IRIS=len(base)
print('IRIS conservés :',NOMBRE_IRIS,'; IRIS du focus :',int(base['J13C_DANS_FOCUS'].sum()))




IRIS conservés : 5264 ; IRIS du focus : 37


In [57]:
#Préparer une table de points et ses identifiants

def paire_coordonnees(v,ordre):
    if pd.isna(v):return (np.nan,np.nan)
    if ordre not in {'LAT_LON','LON_LAT'}:raise ValueError('Ordre de coordonnées non reconnu.')
    try:
        valeur=json.loads(str(v))
        if isinstance(valeur,dict):return float(valeur['lon']),float(valeur['lat'])
        if not isinstance(valeur,(list,tuple)):raise ValueError()
    except (ValueError,KeyError,TypeError):
        s=str(v).strip().strip('()[]')
        morceaux=s.split(';') if ';' in s else s.split(',')
        try:valeur=[float(c.strip().replace(',','.')) for c in morceaux]
        except ValueError:return (np.nan,np.nan)
    if len(valeur)!=2:return (np.nan,np.nan)
    try:a,b=map(float,valeur)
    except (ValueError,TypeError):return (np.nan,np.nan)
    return (b,a) if ordre=='LAT_LON' else (a,b)

def preparer_source(source,cfg):
    ap=apercu_source(cfg);champs=list(ap.columns)
    id_alias=(['CLE_EQUIPEMENT','ID_EQUIPEMENT','IDENTIFIANT','SIRET','ID'] if source!='TRANSPORTS'
              else ['ARRID','ARR_ID','STOP_ID','ID_REF_A','ID_ARRET','ZDE_ID','ID_ZDE','ID_EQUIPEMENT','ID'])
    cat_alias=(['CATEGORIE_RESTAURATION'] if source=='RESTAURATION'
               else ['GROUPE_EQUIPEMENT','GROUPE','LIBELLE_TYPEQU','TYPEQU','CATEGORIE'] if source=='EQUIPEMENTS'
               else ['MODE','MODE_TRANSPORT','TRANSPORTMODE','GROUPE_EQUIPEMENT','CATEGORIE','GROUPE'])
    nom_alias=(['NOM_RESTAURANT','NOM_ETABLISSEMENT','NOMRS','CNOMRS','NOM','NAME'] if source!='TRANSPORTS'
              else ['ARRNAME','ARR_NAME','STOP_NAME','NOM_ARRET','NOM_GARE','NOM_ETABLISSEMENT','NOM','NAME'])
    cid=choisir(champs,cfg['id'],id_alias,source+' — identifiant',priorite=True)
    if source=='TRANSPORTS' and cid is None:
        exporter_csv(pd.DataFrame({'COLONNE':champs}),DOSSIER_INTERIM/'entetes_transports_j13c.csv')
        candidats=[c for c in champs if any(m in c for m in ['ID','IDENTIFIANT','CODE'])]
        raise ValueError('TRANSPORTS : aucune colonne d’identifiant reconnue. Candidats : '+str(candidats)+
            '. Renseigner SOURCES[TRANSPORTS][id] au bloc 3 d’après la notice et les en-têtes.')
    cn=choisir(champs,cfg['nom'],nom_alias,source+' — nom',priorite=True)
    cc=choisir(champs,cfg['categorie'],cat_alias,source+' — catégorie',priorite=True)
    com=choisir(champs,cfg['commune'],['DEPCOM','CODGEO','CODE_COMMUNE','COM','INSEE_COM'],source+' — commune',priorite=True)
    ci=choisir(champs,cfg.get('type_identifiant'),['TYPE_IDENTIFIANT_SOURCE'],source+' — nature de la clé')
    geographique=isinstance(ap,gpd.GeoDataFrame)
    cx=cy=cp=None
    if not geographique:
        cx=choisir(champs,cfg['x'],['LONGITUDE','STOP_LON','LON','LNG','X','X_LAMBERT93','LAMBERT_X'],source+' — X/longitude')
        cy=choisir(champs,cfg['y'],['LATITUDE','STOP_LAT','LAT','Y','Y_LAMBERT93','LAMBERT_Y'],source+' — Y/latitude')
        if cx is None or cy is None:
            cp=choisir(champs,cfg['coordonnees'],['GEO_POINT_2D','ARRGEOPOINT','GEO_POINT','GEOPOS','COORDONNEES','COORDINATES'],source+' — paire de coordonnées',obligatoire=True)
        if cfg['crs'] is None:raise ValueError(source+' : préciser le CRS réel des coordonnées.')
    garder={c for c in [cid,cn,cc,com,cx,cy,cp,ci,'TYPEQU','TYPERESTO','LIGNE_BPE_ORIGINALE','FICHIER_ORIGINAL','LIGNE_SOURCE_ORIGINALE','SIRET','X_SOURCE_BRUT','Y_SOURCE_BRUT','COORDONNEES_SOURCE_BRUTES'] if c is not None and c in champs}
    p=Path(cfg['fichier'])
    if p.suffix.lower()=='.csv':
        sep,enc=dialecte_csv(p)
        blocs=[entetes(b) for b in pd.read_csv(p,sep=sep,encoding=enc,dtype='string',low_memory=False,
            usecols=lambda c:normaliser_nom(c) in garder,chunksize=100_000)]
        table=pd.concat(blocs,ignore_index=True) if blocs else pd.DataFrame(columns=list(garder))
    elif p.suffix.lower()=='.xlsx':
        table=entetes(pd.read_excel(p,sheet_name=cfg['feuille'],engine='openpyxl',dtype='string',
            usecols=lambda c:normaliser_nom(c) in garder))
    else:
        table=entetes(gpd.read_file(p,**({'layer':cfg['couche']} if cfg['couche'] else {})))
    table=table.reset_index(drop=True)
    table['_LIGNE']=np.arange(len(table))+1
    nb_lues=len(table)
    # Une table BPE générale est filtrée explicitement ; les autres données restent dans leur source.
    if 'TYPEQU' in table:
        resto=texte(table['TYPEQU']).str.upper().eq('A504').fillna(False)
        if source=='RESTAURATION':table=table[resto].copy()
        elif source=='EQUIPEMENTS':table=table[~resto].copy()
    if table.empty:raise ValueError(source+' : table vide après sélection. Vérifier le fichier de détail.')
    table=table.reset_index(drop=True)
    out=pd.DataFrame(index=table.index)
    out['SOURCE']=source
    identifiants=texte(table[cid]) if cid else pd.Series(pd.NA,index=table.index,dtype='string')
    if cid=='SIRET':
        identifiants=identifiants.where(identifiants.str.fullmatch(r'\d{14}',na=False) & identifiants.ne('0'*14))
    manquant=identifiants.isna()
    if manquant.any() and not cfg['autoriser_id_ligne']:
        diagnostic=table.loc[manquant].copy()
        if isinstance(diagnostic,gpd.GeoDataFrame):diagnostic=diagnostic.drop(columns=diagnostic.geometry.name)
        diagnostic['COLONNE_IDENTIFIANT_CHOISIE']=cid
        diagnostic['FICHIER_SOURCE']=str(p)
        fichier_diagnostic=DOSSIER_INTERIM/f'identifiants_absents_{source.lower()}_j13c.csv'
        exporter_csv(diagnostic,fichier_diagnostic)
        raise ValueError(source+' : '+str(int(manquant.sum()))+' identifiant(s) absent(s) sur '+str(len(table))+
            ' observations ; colonne choisie : '+str(cid)+'. Voir '+str(fichier_diagnostic)+
            '. Vérifier la colonne au bloc 3 et les lignes source ; aucun arrêt n’a été supprimé.')
    out['ID_SOURCE']=identifiants.fillna('LIGNE_'+table['_LIGNE'].astype(str))
    out['TYPE_IDENTIFIANT']=np.where(manquant,'CLE_DE_LIGNE','IDENTIFIANT_SOURCE')
    if ci:
        nature=texte(table[ci])
        if not nature.isin(['IDENTIFIANT_SOURCE','CLE_DE_LIGNE']).all():
            raise ValueError(source+' : nature de l’identifiant préparé non reconnue.')
        if not cfg['autoriser_id_ligne'] and nature.eq('CLE_DE_LIGNE').any():
            raise ValueError(source+' : identifiant de ligne insuffisant pour l’unité physique attendue.')
        out['TYPE_IDENTIFIANT']=np.where(manquant,'CLE_DE_LIGNE',nature)
    out['CLE_POINT']=source+'::'+out['ID_SOURCE']
    out['LIGNE_SOURCE']=table['_LIGNE']
    out['NOM']=texte(table[cn]) if cn else pd.Series(pd.NA,index=table.index,dtype='string')
    if cc:
        categorie=texte(table[cc])
    elif source=='RESTAURATION' and 'TYPERESTO' in table:
        categorie=texte(table['TYPERESTO']).str.upper().map({'5610A':'TRADITIONNELLE','5610B':'CAFETERIA_LIBRE_SERVICE','5610C':'RAPIDE'})
    else:categorie=pd.Series(pd.NA,index=table.index,dtype='string')
    out['CATEGORIE']=categorie.fillna('TYPE_INCONNU' if source=='RESTAURATION' else 'NON_RENSEIGNE')
    out['CATEGORIE_CODE']=out['CATEGORIE'].map(normaliser_nom)
    for c in ['TYPEQU','LIGNE_BPE_ORIGINALE','FICHIER_ORIGINAL','LIGNE_SOURCE_ORIGINALE','SIRET']:
        if c in table:out[c]=texte(table[c])
    out['COMMUNE_SOURCE']=codes(table[com],5) if com else pd.Series(pd.NA,index=table.index,dtype='string')
    if geographique:
        if table.crs is None:
            if cfg['crs'] is None:raise ValueError(source+' : CRS géographique absent.')
            table=table.set_crs(cfg['crs'])
        non_points=table.geometry.notna() & ~table.geometry.geom_type.eq('Point')
        if non_points.any():raise ValueError(source+' : géométrie de Point attendue, pas des lignes ou polygones.')
        geo=table.to_crs(CRS_TRAVAIL).geometry
        valide=geo.notna() & ~geo.is_empty & geo.is_valid
    else:
        if cp:
            xy=[paire_coordonnees(v,cfg['ordre_coordonnees']) for v in table[cp]]
            x=pd.Series([v[0] for v in xy],index=table.index);y=pd.Series([v[1] for v in xy],index=table.index)
        else:x=nombre(table[cx]);y=nombre(table[cy])
        valide=x.notna() & y.notna()
        if str(cfg['crs']).upper() in {'4326','EPSG:4326'}:valide &= x.between(-180,180) & y.between(-90,90)
        geo=gpd.GeoSeries([None]*len(out),index=out.index,crs=CRS_TRAVAIL)
        if valide.any():
            transformes=gpd.GeoSeries(gpd.points_from_xy(x[valide],y[valide]),index=out.index[valide],crs=cfg['crs']).to_crs(CRS_TRAVAIL)
            geo.loc[valide]=transformes
    out['X_SOURCE_BRUT']=texte(table[cx]) if not geographique and cx else pd.Series(pd.NA,index=table.index,dtype='string')
    out['Y_SOURCE_BRUT']=texte(table[cy]) if not geographique and cy else pd.Series(pd.NA,index=table.index,dtype='string')
    out['COORDONNEES_SOURCE_BRUTES']=texte(table[cp]) if cp else pd.Series(pd.NA,index=table.index,dtype='string')
    for c in ['X_SOURCE_BRUT','Y_SOURCE_BRUT','COORDONNEES_SOURCE_BRUTES']:
        if c in table:out[c]=texte(table[c])
    out=gpd.GeoDataFrame(out,geometry=geo,crs=CRS_TRAVAIL)
    out['X_2154']=out.geometry.x;out['Y_2154']=out.geometry.y
    valide=valide & np.isfinite(out['X_2154']) & np.isfinite(out['Y_2154'])
    out.loc[~valide,'geometry']=None
    out['COORDONNEES_VALIDES']=valide
    repetitions=out[out['CLE_POINT'].duplicated(keep=False)].drop(columns='geometry')
    exporter_csv(repetitions,DOSSIER_INTERIM/f'doublons_{source.lower()}_j13c.csv')
    # Catégorie et position : éléments nécessaires au comptage. Conflits de noms tracés séparément.
    canonique=out.drop_duplicates(['CLE_POINT','X_2154','Y_2154','CATEGORIE_CODE'])
    conflits=canonique[canonique['CLE_POINT'].duplicated(keep=False)]
    if not conflits.empty:
        exporter_csv(conflits.drop(columns='geometry'),DOSSIER_INTERIM/f'conflits_identifiants_{source.lower()}_j13c.csv')
        raise ValueError(source+' : même identifiant avec positions/catégories différentes. Consulter conflits_identifiants.')
    noms=out.groupby('CLE_POINT')['NOM'].first()
    uniques=canonique.copy();uniques['NOM']=uniques['CLE_POINT'].map(noms)
    uniques=uniques.reset_index(drop=True)
    STATISTIQUES_SOURCES.append({'SOURCE':source,'NB_LIGNES_LUES':nb_lues,'NB_APRES_SELECTION':len(out),
        'NB_DOUBLONS_SUPPRIMES':len(out)-len(uniques),'NB_POINTS_UNIQUES':len(uniques),
        'NB_CLES_DE_LIGNE':int(uniques['TYPE_IDENTIFIANT'].eq('CLE_DE_LIGNE').sum()),
        'NB_COORDONNEES_VALIDES':int(uniques['COORDONNEES_VALIDES'].sum()),'UNITE':cfg['unite']})
    LECTURES.append({'SOURCE':source,'FICHIER':str(p),'COL_ID':cid,'COL_NOM':cn,'COL_CATEGORIE':cc,
        'COL_X':cx,'COL_Y':cy,'COL_COORDONNEES':cp,'COL_COMMUNE':com,'CRS_TRAVAIL':CRS_TRAVAIL,
        'CRS_SOURCE':str(table.crs) if geographique else str(cfg['crs']),
        'ANNEE_SOURCE':cfg['annee_source'],'DATE_SOURCE':cfg['date_source'],'UNITE':cfg['unite']})
    return uniques


In [58]:

from contextlib import contextmanager
from zipfile import ZipFile

def analyser_entete_bpe(flux):
    extrait=flux.read(32768)
    import codecs
    try:chaine=codecs.getincrementaldecoder('utf-8-sig')().decode(extrait,final=False);enc='utf-8-sig'
    except UnicodeDecodeError:chaine=extrait.decode('cp1252');enc='cp1252'
    ligne=chaine.splitlines()[0] if chaine.splitlines() else ''
    choix=[]
    for sep in [';',',','\t']:
        cols=[normaliser_nom(c) for c in next(csv.reader([ligne],delimiter=sep))]
        if {'DEPCOM','TYPEQU'}<=set(cols):choix.append((sep,enc,cols))
    if len(choix)!=1:raise ValueError('Détail BPE attendu : DEPCOM et TYPEQU séparés ; vérifier fichier et séparateur.')
    return choix[0]

@contextmanager
def ouvrir_detail_bpe(path,membre=None):
    path=Path(path)
    if not path.is_file():raise FileNotFoundError('Détail BPE absent : '+str(path))
    if path.suffix.lower()=='.csv':
        with path.open('rb') as f:sep,enc,cols=analyser_entete_bpe(f)
        with path.open('rb') as f:yield f,sep,enc,cols,None
    elif path.suffix.lower()=='.zip':
        with ZipFile(path) as z:
            possibles=[]
            for item in z.infolist():
                if not item.filename.lower().endswith('.csv'):continue
                if membre is not None and item.filename!=membre:continue
                try:
                    with z.open(item.filename) as f:info=analyser_entete_bpe(f)
                except ValueError:continue
                possibles.append((item.filename,*info))
            if len(possibles)!=1:
                raise ValueError('ZIP : choisir un seul CSV détail compatible au paramètre MEMBRE_CSV_BPE. Candidats : '+str([x[0] for x in possibles]))
            nom,sep,enc,cols=possibles[0]
            with z.open(nom) as f:yield f,sep,enc,cols,nom
    else:raise ValueError('Détail BPE brut attendu en CSV ou ZIP contenant le CSV ; utiliser une table de points préparée pour les autres formats.')

def charger_nomenclature():
    if FICHIER_NOMENCLATURE_BPE is None:return pd.DataFrame(columns=['CODE','LIBELLE'])
    p=Path(FICHIER_NOMENCLATURE_BPE)
    sep,enc=dialecte_csv(p)
    t=entetes(pd.read_csv(p,sep=sep,encoding=enc,dtype='string'))
    exporter_csv(pd.DataFrame({'COLONNE':t.columns}),DOSSIER_INTERIM/'entetes_nomenclature_j13c.csv')
    print('Colonnes nomenclature :',t.columns.tolist())
    c=choisir(t.columns,COLONNE_CODE_NOMENCLATURE,['TYPEQU','CODE','CODE_TYPEQU'],
        'Nomenclature — code',obligatoire=True)
    l=choisir(t.columns,COLONNE_LIBELLE_NOMENCLATURE,
        ['LIBELLE','LIBELLE_TYPEQU','LIB_TYPEQU','LIBELLE_EQUIPEMENT','LIBEQU'],
        'Nomenclature — libellé',obligatoire=True)
    t=pd.DataFrame({'CODE':texte(t[c]).str.upper(),'LIBELLE':texte(t[l])}).dropna()
    if t['CODE'].duplicated().any():raise ValueError('Nomenclature : un libellé unique est attendu par code.')
    return t

def lire_blocs_bpe_stricts(flux,sep,enc,colonnes,colonnes_gardees,taille_bloc):
    import io
    lecteur=csv.reader(io.TextIOWrapper(flux,encoding=enc,newline=''),delimiter=sep,strict=True)
    try:
        entete_lue=[normaliser_nom(c) for c in next(lecteur)]
        if entete_lue!=colonnes:raise ValueError('Détail BPE : en-tête différent entre inspection et lecture.')
        indices=[i for i,c in enumerate(colonnes) if c in colonnes_gardees]
        noms=[colonnes[i] for i in indices]
        lignes=[]
        for ligne in lecteur:
            if not ligne:continue
            if len(ligne)!=len(colonnes):
                raise pd.errors.ParserError('Détail BPE : ligne physique '+str(lecteur.line_num)+
                    ', '+str(len(ligne))+' champs au lieu de '+str(len(colonnes))+'.')
            lignes.append([ligne[i] for i in indices])
            if len(lignes)>=taille_bloc:
                yield pd.DataFrame(lignes,columns=noms,dtype='string').replace('',pd.NA)
                lignes=[]
        if lignes:yield pd.DataFrame(lignes,columns=noms,dtype='string').replace('',pd.NA)
    except csv.Error as exc:
        raise pd.errors.ParserError('Détail BPE : CSV mal formé près de la ligne physique '+
            str(lecteur.line_num)+'. '+str(exc)) from exc

def extraire_bpe_idf(path,destination,taille_bloc=50_000,membre=None):
    if taille_bloc<1:raise ValueError('TAILLE_BLOC_BPE doit être positive.')
    if Path(path).resolve()==Path(destination).resolve():raise ValueError('La sortie doit être différente du fichier brut.')
    temporaire=destination.with_name(destination.stem+'_temporaire.csv')
    lus=0;gardes=0;restos=0;communes_absentes=0;premier=True
    colonnes_gardees={'DEPCOM','TYPEQU','TYPERESTO','NOMRS','CNOMRS','NOM','NOM_ETABLISSEMENT',
        'NOM_RESTAURANT','LONGITUDE','LATITUDE','SIRET','CLE_EQUIPEMENT','ID_EQUIPEMENT','IDENTIFIANT',
        'ID','GROUPE_EQUIPEMENT','GROUPE','LIBELLE_TYPEQU','CATEGORIE_RESTAURATION','AN'}
    try:
        with ouvrir_detail_bpe(path,membre) as (flux,sep,enc,cols,membre_lu):
            if len(cols)!=len(set(cols)):raise ValueError('Détail BPE : en-têtes normalisés dupliqués.')
            if not {'LONGITUDE','LATITUDE'}<=set(cols):
                raise ValueError('Extraction BPE : LONGITUDE et LATITUDE attendues. Préparer séparément un fichier utilisant un autre CRS.')
            exporter_csv(pd.DataFrame({'COLONNE':cols}),DOSSIER_INTERIM/'entetes_bpe_detail_j13c.csv')
            ident=next((c for c in ['CLE_EQUIPEMENT','ID_EQUIPEMENT','IDENTIFIANT','ID'] if c in cols),None)
            lecteur=lire_blocs_bpe_stricts(flux,sep,enc,cols,colonnes_gardees,taille_bloc)
            for b in lecteur:
                b=entetes(b)
                positions=pd.Series(np.arange(lus+1,lus+len(b)+1),index=b.index,dtype='Int64')
                lus+=len(b)
                b['DEPCOM']=codes(b['DEPCOM'],5)
                communes_absentes+=int(b['DEPCOM'].isna().sum())
                b=b[b['DEPCOM'].str[:2].isin(DEPARTEMENTS_IDF)].copy()
                if b.empty:continue
                b['TYPEQU']=texte(b['TYPEQU']).str.upper()
                if 'AN' in b:
                    annees=nombre(b['AN']).dropna()
                    if not annees.eq(ANNEE_BPE).all():
                        raise ValueError('BPE : AN ne correspond pas à ANNEE_BPE='+str(ANNEE_BPE)+'. Vérifier source et millésime.')
                if 'CATEGORIE_RESTAURATION' not in b:
                    typ=texte(b['TYPERESTO']).str.upper().str.replace(r'[^A-Z0-9]','',regex=True) if 'TYPERESTO' in b else pd.Series(pd.NA,index=b.index,dtype='string')
                    b['CATEGORIE_RESTAURATION']=typ.map({'5610A':'TRADITIONNELLE','5610B':'CAFETERIA_LIBRE_SERVICE','5610C':'RAPIDE'}).fillna('TYPE_INCONNU')
                b['LIGNE_BPE_ORIGINALE']=positions.loc[b.index].astype('string')
                ids=texte(b[ident]) if ident else pd.Series(pd.NA,index=b.index,dtype='string')
                b['TYPE_IDENTIFIANT_SOURCE']=np.where(ids.notna(),'IDENTIFIANT_SOURCE','CLE_DE_LIGNE')
                b['CLE_EQUIPEMENT']=ids.fillna('BPE'+str(ANNEE_BPE)+'_LIGNE_'+b['LIGNE_BPE_ORIGINALE'])
                b['FICHIER_ORIGINAL']=str(Path(path))
                if not NOMENCLATURE.empty:
                    correspondance=NOMENCLATURE.set_index('CODE')['LIBELLE']
                    b['LIBELLE_TYPEQU']=b['TYPEQU'].map(correspondance)
                elif 'LIBELLE_TYPEQU' not in b:b['LIBELLE_TYPEQU']=pd.NA
                if 'GROUPE_EQUIPEMENT' not in b:
                    b['GROUPE_EQUIPEMENT']=(texte(b['GROUPE']) if 'GROUPE' in b else texte(b['LIBELLE_TYPEQU']))
                b['GROUPE_EQUIPEMENT']=texte(b['GROUPE_EQUIPEMENT']).fillna('Type BPE '+b['TYPEQU'].fillna('non renseigné'))
                b['LIBELLE_CATEGORIE_DOCUMENTE']=~b['GROUPE_EQUIPEMENT'].str.startswith('Type BPE ',na=False)
                b.to_csv(temporaire,index=False,encoding='utf-8-sig',mode='w' if premier else 'a',header=premier)
                premier=False;gardes+=len(b);restos+=int(b['TYPEQU'].eq('A504').sum())
        if premier:raise ValueError('Aucune observation IDF dans le détail BPE sélectionné.')
        temporaire.replace(destination)
    except Exception:
        if temporaire.exists():temporaire.unlink()
        raise
    bilan={'FICHIER_ORIGINAL':str(path),'MEMBRE_ZIP':membre_lu,'NB_LIGNES_LUES':lus,
        'NB_OBSERVATIONS_IDF':gardes,'NB_RESTAURANTS_IDF_A504':restos,
        'NB_COMMUNES_ABSENTES_SOURCE':communes_absentes,'TAILLE_BLOC':taille_bloc,'SORTIE':str(destination),
        'IDENTIFIANT_ORIGINAL':ident or 'clé de ligne de cette version BPE'}
    exporter_csv(pd.DataFrame([bilan]),DOSSIER_INTERIM/'bilan_preparation_bpe_j13c.csv')
    PREPARATIONS_EFFECTUEES.append({'ETAPE':'EXTRACTION_BPE_IDF',**bilan})
    return bilan


NOMENCLATURE=charger_nomenclature()
sources_bpe_a_construire=[nom for nom in ['RESTAURATION','EQUIPEMENTS']
    if SOURCES[nom]['actif'] and SOURCES[nom]['fichier'] is None]
if sources_bpe_a_construire and FICHIER_BPE_DETAIL is not None and Path(FICHIER_BPE_DETAIL).is_file():
    bilan_bpe=extraire_bpe_idf(FICHIER_BPE_DETAIL,FICHIER_BPE_IDF,TAILLE_BLOC_BPE,MEMBRE_CSV_BPE)
    for source in sources_bpe_a_construire:
        SOURCES[source].update({'fichier':FICHIER_BPE_IDF,'id':'CLE_EQUIPEMENT',
            'categorie':'CATEGORIE_RESTAURATION' if source=='RESTAURATION' else 'GROUPE_EQUIPEMENT',
            'x':'LONGITUDE','y':'LATITUDE','crs':4326,'commune':'DEPCOM',
            'type_identifiant':'TYPE_IDENTIFIANT_SOURCE'})
    display(pd.DataFrame([bilan_bpe]))
else:print('BPE : entrées déjà préparées, sources désactivées ou détail brut à renseigner au bloc 3.')





,FICHIER_ORIGINAL,MEMBRE_ZIP,NB_LIGNES_LUES,NB_OBSERVATIONS_IDF,NB_RESTAURANTS_IDF_A504,NB_COMMUNES_ABSENTES_SOURCE,TAILLE_BLOC,SORTIE,IDENTIFIANT_ORIGINAL
0,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,None,2921770,516711,49169,0,50000,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,clé de ligne de cette version BPE


In [59]:
#Préparer les arrets IDFM

def adapter_schema_idfm(cfg):
    ap=apercu_source(cfg)
    cols=set(ap.columns)
    if 'ARRID' in cols and cfg['id'] is None:cfg['id']='ARRID'
    if 'ARRNAME' in cols and cfg['nom'] is None:cfg['nom']='ARRNAME'
    if not isinstance(ap,gpd.GeoDataFrame) and cfg['x'] is None and cfg['y'] is None:
        if {'ARRXEPSG2154','ARRYEPSG2154'}<=cols:
            cfg.update({'x':'ARRXEPSG2154','y':'ARRYEPSG2154','crs':2154})
        elif 'ARRGEOPOINT' in cols and cfg['coordonnees'] is None:
            cfg.update({'coordonnees':'ARRGEOPOINT','crs':4326})
    candidats=[c for c in ap.columns if any(m in c for m in ['ID','IDENTIFIANT','CODE'])]
    exporter_csv(pd.DataFrame({'COLONNE_CANDIDATE':candidats}),DOSSIER_INTERIM/'candidats_identifiants_idfm_j13c.csv')
    print('IDFM — colonnes d’identifiant candidates :',candidats)
    print('IDFM — identifiant déclaré :',cfg['id'],'; unité :',cfg['unite'])

def preparer_cle_idfm_par_ligne(cfg):
    if not COLONNES_CLE_TRANSPORTS:return
    colonnes=[normaliser_nom(c) for c in COLONNES_CLE_TRANSPORTS]
    if len(colonnes)<2:raise ValueError('Clé composite : déclarer au moins deux composants documentés.')
    ap=apercu_source(cfg)
    if set(colonnes)-set(ap.columns):raise ValueError('Composants IDFM absents : '+str(sorted(set(colonnes)-set(ap.columns))))
    path=Path(cfg['fichier'])
    if path.suffix.lower()=='.csv':
        sep,enc=dialecte_csv(path);t=entetes(pd.read_csv(path,sep=sep,encoding=enc,dtype='string',low_memory=False))
    elif path.suffix.lower()=='.xlsx':t=entetes(pd.read_excel(path,sheet_name=cfg['feuille'],dtype='string'))
    else:t=entetes(gpd.read_file(path,**({'layer':cfg['couche']} if cfg['couche'] else {})))
    composants=t[colonnes].apply(texte)
    manquant=composants.isna().any(axis=1)
    if manquant.any():
        d=t.loc[manquant].copy()
        if isinstance(d,gpd.GeoDataFrame):d=d.drop(columns=d.geometry.name)
        exporter_csv(d,DOSSIER_INTERIM/'composants_cle_idfm_absents_j13c.csv')
        raise ValueError('IDFM : composants de clé composite absents ; voir composants_cle_idfm_absents_j13c.csv.')
    # JSON encodes components without delimiter collisions, preserving their order.
    if 'FICHIER_ORIGINAL' not in t:t['FICHIER_ORIGINAL']=str(path)
    if 'LIGNE_SOURCE_ORIGINALE' not in t:t['LIGNE_SOURCE_ORIGINALE']=np.arange(len(t))+1
    t['ID_COMPOSITE_IDFM']=[json.dumps(list(r),ensure_ascii=False,separators=(',',':'))
        for r in composants.itertuples(index=False,name=None)]
    if isinstance(t,gpd.GeoDataFrame):
        destination=DOSSIER_INTERIM/'arrets_idfm_cle_composite_j13c.gpkg'
        temporaire=destination.with_name(destination.stem+'_temporaire.gpkg')
        if temporaire.exists():temporaire.unlink()
        t.to_file(temporaire,layer='arrets',driver='GPKG',index=False)
        temporaire.replace(destination);cfg.update({'fichier':destination,'couche':'arrets'})
    else:
        destination=DOSSIER_INTERIM/'arrets_idfm_cle_composite_j13c.csv'
        exporter_csv(t,destination);cfg.update({'fichier':destination,'couche':None})
    cfg['id']='ID_COMPOSITE_IDFM'
    cfg['unite']='Observation de transport définie par la clé composite '+', '.join(colonnes)+' ; ne représente pas automatiquement une gare unique'
    PREPARATIONS_EFFECTUEES.append({'ETAPE':'CLE_COMPOSITE_IDFM','SOURCE':'TRANSPORTS','FICHIER_ORIGINAL':str(path),
        'SORTIE':str(destination),'COMPOSANTS':colonnes,'UNITE':cfg['unite'],'NB_OBSERVATIONS':len(t)})

def enregistrer_entree_points(source,cfg,destination):
    original_path=Path(cfg['fichier'])
    if original_path.resolve()==destination.resolve():
        ap=apercu_source(cfg)
        requis={'ID_EQUIPEMENT','NOM_ETABLISSEMENT','GROUPE_EQUIPEMENT','X_2154','Y_2154','DEPCOM','TYPE_IDENTIFIANT_SOURCE'}
        if requis-set(ap.columns):
            raise ValueError(source+' : entrée préparée incomplète : '+str(sorted(requis-set(ap.columns))))
        cfg.update({'fichier':destination,'couche':None,'feuille':0,'id':'ID_EQUIPEMENT',
            'nom':'NOM_ETABLISSEMENT','categorie':'GROUPE_EQUIPEMENT','x':'X_2154','y':'Y_2154',
            'coordonnees':None,'crs':2154,'commune':'DEPCOM','type_identifiant':'TYPE_IDENTIFIANT_SOURCE'})
        print(source,': entrée normalisée réutilisée sans écrasement :',destination)
        return
    observations=preparer_source(source,cfg)
    doublons_originaux=DOSSIER_INTERIM/f'doublons_{source.lower()}_j13c.csv'
    rapport_preparation=DOSSIER_INTERIM/f'doublons_preparation_{source.lower()}_j13c.csv'
    if doublons_originaux.is_file():rapport_preparation.write_bytes(doublons_originaux.read_bytes())
    wgs=observations.to_crs(4326)
    t=pd.DataFrame({'ID_EQUIPEMENT':observations['ID_SOURCE'],
        'TYPE_IDENTIFIANT_SOURCE':observations['TYPE_IDENTIFIANT'],
        'NOM_ETABLISSEMENT':observations['NOM'],'GROUPE_EQUIPEMENT':observations['CATEGORIE'],
        'LONGITUDE':wgs.geometry.x,'LATITUDE':wgs.geometry.y,
        'X_2154':observations['X_2154'],'Y_2154':observations['Y_2154'],'DEPCOM':observations['COMMUNE_SOURCE'],
        'FICHIER_ORIGINAL':observations['FICHIER_ORIGINAL'] if 'FICHIER_ORIGINAL' in observations else str(original_path),
        'LIGNE_SOURCE_ORIGINALE':observations['LIGNE_SOURCE_ORIGINALE'] if 'LIGNE_SOURCE_ORIGINALE' in observations else observations['LIGNE_SOURCE'],
        'X_SOURCE_BRUT':observations['X_SOURCE_BRUT'],'Y_SOURCE_BRUT':observations['Y_SOURCE_BRUT'],
        'COORDONNEES_SOURCE_BRUTES':observations['COORDONNEES_SOURCE_BRUTES']})
    for c in ['TYPEQU','LIGNE_BPE_ORIGINALE','SIRET']:
        if c in observations:t[c]=observations[c]
    tmp=destination.with_name(destination.stem+'_temporaire.csv')
    try:
        exporter_csv(t,tmp)
        tmp.replace(destination)
    except Exception:
        if tmp.exists():tmp.unlink()
        raise
    PREPARATIONS_EFFECTUEES.append({'ETAPE':'NORMALISATION_POINTS','SOURCE':source,
        'FICHIER_ORIGINAL':str(original_path),'SORTIE':str(destination),'NB_POINTS':len(t),
        'NB_COORDONNEES_VALIDES':int(observations['COORDONNEES_VALIDES'].sum()),
        'NB_CLES_DE_LIGNE':int(observations['TYPE_IDENTIFIANT'].eq('CLE_DE_LIGNE').sum()),
        'RAPPORT_DOUBLONS_ORIGINAUX':str(rapport_preparation),'LECTURE_ORIGINALE':dict(LECTURES[-1]),'BILAN_ORIGINAL':dict(STATISTIQUES_SOURCES[-1])})
    cfg.update({'fichier':destination,'couche':None,'feuille':0,'id':'ID_EQUIPEMENT',
        'nom':'NOM_ETABLISSEMENT','categorie':'GROUPE_EQUIPEMENT','x':'X_2154','y':'Y_2154',
        'coordonnees':None,'crs':2154,'commune':'DEPCOM','type_identifiant':'TYPE_IDENTIFIANT_SOURCE'})
    print(source,': CSV préparé :',destination,';',len(t),'points.')

cfg=SOURCES['TRANSPORTS']
if cfg['actif'] and cfg['fichier'] is not None and Path(cfg['fichier']).is_file():
    adapter_schema_idfm(cfg)
    preparer_cle_idfm_par_ligne(cfg)
    enregistrer_entree_points('TRANSPORTS',cfg,FICHIER_ARRETS_PREPARES)
else:print('TRANSPORTS : sélectionner les points d’arrêt au bloc 3, ou source explicitement désactivée.')



IDFM — colonnes d’identifiant candidates : ['ARRID', 'ZDAID']
IDFM — identifiant déclaré : ARRID ; unité : Point de transport selon l’identifiant et la granularité du fichier IDFM sélectionné
TRANSPORTS : CSV préparé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13c\entrees\arrets_idfm_idf_j13c.csv ; 37956 points.


In [60]:
#Construire les restaurants et les équipements depuis la BPE

for source,destination in [('RESTAURATION',FICHIER_RESTAURANTS_PREPARES),
                           ('EQUIPEMENTS',FICHIER_EQUIPEMENTS_PREPARES)]:
    cfg=SOURCES[source]
    if cfg['actif'] and cfg['fichier'] is not None and Path(cfg['fichier']).is_file():
        ap=apercu_source(cfg)
        if 'TYPEQU' not in ap.columns:
            print(source,': TYPEQU absente ; vérifier le périmètre dans la préparation de cette entrée.')
        enregistrer_entree_points(source,cfg,destination)
    else:print(source,': détail BPE ou entrée préparée à renseigner au bloc 3, ou source désactivée.')

cfg=SOURCES['EQUIPEMENTS']
if cfg['actif'] and cfg['fichier'] is not None and Path(cfg['fichier']).is_file():
    libelles=pd.read_csv(cfg['fichier'],dtype='string',encoding='utf-8-sig',usecols=['GROUPE_EQUIPEMENT'])
    libelles['LIBELLE_DOCUMENTE']=~(libelles['GROUPE_EQUIPEMENT'].str.startswith('Type BPE ',na=False) |
        libelles['GROUPE_EQUIPEMENT'].str.fullmatch(r'[A-Z]\d{3}',na=False) |
        libelles['GROUPE_EQUIPEMENT'].isin(['NON_RENSEIGNE','TYPE_INCONNU']))
    qualite_libelles=libelles.groupby(['GROUPE_EQUIPEMENT','LIBELLE_DOCUMENTE']).size().reset_index(name='NB_OBSERVATIONS')
    exporter_csv(qualite_libelles,DOSSIER_INTERIM/'qualite_libelles_equipements_j13c.csv')
    print('Équipements avec catégorie restant à libeller :',int((~libelles['LIBELLE_DOCUMENTE']).sum()))


RESTAURATION : CSV préparé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13c\entrees\restaurants_idf_j13c.csv ; 49169 points.
EQUIPEMENTS : CSV préparé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13c\entrees\equipements_idf_j13c.csv ; 467542 points.
Équipements avec catégorie restant à libeller : 467542


In [61]:
#Vérifier les entrées 

etat=[]
for source,cfg in SOURCES.items():
    fichier=cfg['fichier']
    existe=fichier is not None and Path(fichier).is_file()
    statut='PRETE' if cfg['actif'] and existe else 'NON_CHARGEE' if not cfg['actif'] else 'A_PREPARER'
    etat.append({'SOURCE':source,'ACTIF':cfg['actif'],'STATUT':statut,
        'FICHIER':str(fichier) if fichier is not None else '', 'UNITE':cfg['unite']})
    if cfg['actif'] and existe:
        ap=apercu_source(cfg)
        exporter_csv(pd.DataFrame({'COLONNE':ap.columns}),DOSSIER_INTERIM/f'entetes_{source.lower()}_j13c.csv')
etat=pd.DataFrame(etat)
exporter_csv(etat,DOSSIER_INTERIM/'etat_entrees_j13c.csv')
(DOSSIER_INTERIM/'preparations_sources_j13c.json').write_text(
    json.dumps(PREPARATIONS_EFFECTUEES,ensure_ascii=False,indent=2),encoding='utf-8')
display(etat)
a_preparer=etat.loc[etat['STATUT'].eq('A_PREPARER'),'SOURCE'].tolist()
if a_preparer:
    raise ValueError('J13c : entrées à préparer : '+', '.join(a_preparer)+
        '. Voir etat_entrees_j13c.csv et travail_a_reprendre_j13c.csv. Renseigner les vrais chemins au bloc 3 puis relancer les blocs 1–11.')
print('Entrées prêtes pour l’affectation :',etat.loc[etat['STATUT'].eq('PRETE'),'SOURCE'].tolist())


,SOURCE,ACTIF,STATUT,FICHIER,UNITE
0,RESTAURATION,True,PRETE,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,"Observation de restaurant BPE, dédupliquée sel..."
1,TRANSPORTS,True,PRETE,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,Point de transport selon l’identifiant et la g...
2,EQUIPEMENTS,True,PRETE,C:\Users\almou\OneDrive\GeoMarketing_IDF\data\...,Observation d’équipement complémentaire de la ...


Entrées prêtes pour l’affectation : ['RESTAURATION', 'TRANSPORTS', 'EQUIPEMENTS']


In [62]:
# Charger et dédpupliquer chaque source

points_par_source={}
STATISTIQUES_SOURCES=[];LECTURES=[]
for source,cfg in SOURCES.items():
    if cfg['actif']:
        points_par_source[source]=preparer_source(source,cfg)
        print(source,':',len(points_par_source[source]),'points après déduplication.')
statistiques_sources=pd.DataFrame(STATISTIQUES_SOURCES)
exporter_csv(statistiques_sources,DOSSIER_INTERIM/'qualite_sources_points_j13c.csv')
display(statistiques_sources)


RESTAURATION : 49169 points après déduplication.
TRANSPORTS : 37956 points après déduplication.
EQUIPEMENTS : 467542 points après déduplication.


,SOURCE,NB_LIGNES_LUES,NB_APRES_SELECTION,NB_DOUBLONS_SUPPRIMES,NB_POINTS_UNIQUES,NB_CLES_DE_LIGNE,NB_COORDONNEES_VALIDES,UNITE
0,RESTAURATION,49169,49169,0,49169,49169,47856,"Observation de restaurant BPE, dédupliquée sel..."
1,TRANSPORTS,37956,37956,0,37956,0,37956,Point de transport selon l’identifiant et la g...
2,EQUIPEMENTS,467542,467542,0,467542,467542,446956,Observation d’équipement complémentaire de la ...


In [63]:
# Controler les coordonnées

xmin,ymin,xmax,ymax=iris_polygones.total_bounds
for source,p in points_par_source.items():
    emprise=p['X_2154'].between(xmin,xmax) & p['Y_2154'].between(ymin,ymax)
    p['DANS_EMPRISE_RECTANGULAIRE_IDF']=emprise.fillna(False)
    mauvais=~p['COORDONNEES_VALIDES'] | ~p['DANS_EMPRISE_RECTANGULAIRE_IDF']
    exporter_csv(p.loc[mauvais].drop(columns='geometry'),DOSSIER_INTERIM/f'coordonnees_a_controler_{source.lower()}_j13c.csv')
    print(source,':',int((~p['COORDONNEES_VALIDES']).sum()),'coordonnées absentes/invalides ;',
          int((p['COORDONNEES_VALIDES'] & ~emprise).sum()),'points hors emprise rectangulaire.')


RESTAURATION : 1313 coordonnées absentes/invalides ; 0 points hors emprise rectangulaire.
TRANSPORTS : 0 coordonnées absentes/invalides ; 120 points hors emprise rectangulaire.
EQUIPEMENTS : 20586 coordonnées absentes/invalides ; 1 points hors emprise rectangulaire.


In [64]:
# Affecter les points aux IRIS

def affecter_points(points):
    resultat=points.copy().reset_index(drop=True)
    resultat['IRIS_AFFECTE']=pd.Series(pd.NA,index=resultat.index,dtype='string')
    resultat['STATUT_AFFECTATION']='COORDONNEES_ABSENTES_OU_INVALIDES'
    resultat['NB_IRIS_CANDIDATS']=0
    resultat['IRIS_CANDIDATS']=''
    valides=resultat[resultat['COORDONNEES_VALIDES']][['CLE_POINT','geometry']]
    resultat.loc[resultat['COORDONNEES_VALIDES'],'STATUT_AFFECTATION']='HORS_IRIS_IDF'
    listes=[]
    deja=set()
    for pred,un,several in [('within','INTERIEUR_UNIQUE','INTERIEUR_MULTIPLE'),
                            ('intersects','LIMITE_UNIQUE','LIMITE_MULTIPLE')]:
        selection=valides[~valides['CLE_POINT'].isin(deja)]
        if selection.empty:continue
        joint=gpd.sjoin(selection,iris_polygones[['IRIS','geometry']],how='inner',predicate=pred)
        joint=joint[['CLE_POINT','IRIS']].drop_duplicates()
        groupes=joint.groupby('CLE_POINT')['IRIS'].agg(lambda s:sorted(set(s)))
        correspondances=resultat['CLE_POINT'].map(groupes)
        trouve=correspondances.notna()
        nb=correspondances.loc[trouve].map(len)
        resultat.loc[trouve,'NB_IRIS_CANDIDATS']=nb
        resultat.loc[trouve,'IRIS_CANDIDATS']=correspondances.loc[trouve].map(';'.join)
        resultat.loc[trouve,'STATUT_AFFECTATION']=np.where(nb.eq(1),un,several)
        unique=trouve & resultat['NB_IRIS_CANDIDATS'].eq(1)
        resultat.loc[unique,'IRIS_AFFECTE']=correspondances.loc[unique].map(lambda s:s[0])
        listes.append(joint.rename(columns={'IRIS':'IRIS_CANDIDAT'}).assign(PREDICAT=pred))
        deja.update(groupes.index)
    candidats=(pd.concat(listes,ignore_index=True) if listes else
               pd.DataFrame(columns=['CLE_POINT','IRIS_CANDIDAT','PREDICAT']))
    return resultat,candidats

affectations=[];candidats=[]
for source,p in points_par_source.items():
    resultat,possibles=affecter_points(p)
    affectations.append(resultat);candidats.append(possibles)
points=gpd.GeoDataFrame(pd.concat(affectations,ignore_index=True),geometry='geometry',crs=CRS_TRAVAIL)
candidats_iris=pd.concat(candidats,ignore_index=True)
exporter_csv(candidats_iris,DOSSIER_INTERIM/'candidats_affectation_iris_j13c.csv')
assert points['CLE_POINT'].is_unique


In [65]:
#Lire les points non affectées

bilan_affectations=points.groupby(['SOURCE','STATUT_AFFECTATION']).size().reset_index(name='NB_POINTS')
exporter_csv(bilan_affectations,DOSSIER_INTERIM/'bilan_affectations_initiales_j13c.csv')
non_affectes=points[points['IRIS_AFFECTE'].isna()].copy()
exporter_csv(non_affectes.drop(columns='geometry'),DOSSIER_INTERIM/'points_non_affectes_j13c.csv')
display(bilan_affectations)
display(non_affectes[['SOURCE','ID_SOURCE','NOM','STATUT_AFFECTATION','IRIS_CANDIDATS']].head(15))


,SOURCE,STATUT_AFFECTATION,NB_POINTS
0,EQUIPEMENTS,COORDONNEES_ABSENTES_OU_INVALIDES,20586
1,EQUIPEMENTS,HORS_IRIS_IDF,1
2,EQUIPEMENTS,INTERIEUR_UNIQUE,446955
3,RESTAURATION,COORDONNEES_ABSENTES_OU_INVALIDES,1313
4,RESTAURATION,INTERIEUR_UNIQUE,47856
5,TRANSPORTS,HORS_IRIS_IDF,410
6,TRANSPORTS,INTERIEUR_UNIQUE,37546


,SOURCE,ID_SOURCE,NOM,STATUT_AFFECTATION,IRIS_CANDIDATS
125,RESTAURATION,BPE2025_LIGNE_93506,<NA>,COORDONNEES_ABSENTES_OU_INVALIDES,
478,RESTAURATION,BPE2025_LIGNE_93859,AVENDY,COORDONNEES_ABSENTES_OU_INVALIDES,
584,RESTAURATION,BPE2025_LIGNE_93965,<NA>,COORDONNEES_ABSENTES_OU_INVALIDES,
864,RESTAURATION,BPE2025_LIGNE_94245,OMNIFOOD,COORDONNEES_ABSENTES_OU_INVALIDES,
870,RESTAURATION,BPE2025_LIGNE_94251,<NA>,COORDONNEES_ABSENTES_OU_INVALIDES,
881,RESTAURATION,BPE2025_LIGNE_94262,FAIRY,COORDONNEES_ABSENTES_OU_INVALIDES,
890,RESTAURATION,BPE2025_LIGNE_94271,<NA>,COORDONNEES_ABSENTES_OU_INVALIDES,
1297,RESTAURATION,BPE2025_LIGNE_97831,M.C CAFE,COORDONNEES_ABSENTES_OU_INVALIDES,
1434,RESTAURATION,BPE2025_LIGNE_97968,FOOD FRANCHISE PARIS,COORDONNEES_ABSENTES_OU_INVALIDES,
1469,RESTAURATION,BPE2025_LIGNE_98003,COMME UN DIMANCHE,COORDONNEES_ABSENTES_OU_INVALIDES,


In [66]:
#Traiter les limites avec une correction documentée

ambigu=points['STATUT_AFFECTATION'].isin(['LIMITE_MULTIPLE','INTERIEUR_MULTIPLE'])
modele_limites=DOSSIER_INTERIM/'modele_corrections_limites_j13c.csv'
if not modele_limites.exists():
    modele=points.loc[ambigu,['CLE_POINT','NOM','IRIS_CANDIDATS']].copy()
    for c in ['IRIS_CIBLE','MOTIF','SOURCE_VALIDATION']:modele[c]=''
    modele['VALIDE']=False
    exporter_csv(modele,modele_limites)
corrections_utilisees=pd.DataFrame(columns=['CLE_POINT','IRIS_CIBLE','MOTIF','SOURCE_VALIDATION','VALIDE'])
if FICHIER_CORRECTIONS_LIMITES is not None:
    f=Path(FICHIER_CORRECTIONS_LIMITES)
    if not f.is_file():raise FileNotFoundError('Corrections absentes : '+str(f))
    corrections=entetes(pd.read_csv(f,sep=None,engine='python',dtype='string',encoding='utf-8-sig'))
    requis={'CLE_POINT','IRIS_CIBLE','MOTIF','SOURCE_VALIDATION','VALIDE'}
    if requis-set(corrections):raise ValueError('Corrections : colonnes obligatoires absentes.')
    corrections=corrections[booleen(corrections['VALIDE'],'Corrections — VALIDE')].copy()
    if corrections['CLE_POINT'].isna().any() or not corrections['CLE_POINT'].is_unique:
        raise ValueError('Corrections : clé absente ou dupliquée.')
    corrections['IRIS_CIBLE']=codes(corrections['IRIS_CIBLE'],9)
    if set(corrections['CLE_POINT'])-set(points['CLE_POINT']) or set(corrections['IRIS_CIBLE'])-ATTENDUS:
        raise ValueError('Correction hors des points/IRIS du traitement.')
    for c in ['MOTIF','SOURCE_VALIDATION']:
        if texte(corrections[c]).isna().any():raise ValueError('Correction non documentée : '+c)
    par_cle=points.set_index('CLE_POINT')
    for _,r in corrections.iterrows():
        observation=par_cle.loc[r['CLE_POINT']]
        if observation['STATUT_AFFECTATION'] not in ['LIMITE_MULTIPLE','INTERIEUR_MULTIPLE']:
            raise ValueError('Correction réservée à un point ambigu : '+r['CLE_POINT'])
        if r['IRIS_CIBLE'] not in observation['IRIS_CANDIDATS'].split(';'):
            raise ValueError('Le code choisi ne fait pas partie des candidats : '+r['CLE_POINT'])
        m=points['CLE_POINT'].eq(r['CLE_POINT'])
        points.loc[m,'IRIS_AFFECTE']=r['IRIS_CIBLE']
        points.loc[m,'STATUT_AFFECTATION']='LIMITE_OU_AMBIGUITE_VALIDEE'
    corrections_utilisees=corrections
exporter_csv(corrections_utilisees,DOSSIER_INTERIM/'corrections_limites_utilisees_j13c.csv')
print('Corrections retenues :',len(corrections_utilisees))


Corrections retenues : 0


In [67]:
# Comparer la commune déclarée à la commune spatiale

communes=base.set_index('IRIS')[['CODGEO_SOURCE','CODGEO_ETUDE']]
points['CODGEO_SPATIAL_SOURCE']=points['IRIS_AFFECTE'].map(communes['CODGEO_SOURCE'])
points['CODGEO_SPATIAL_ETUDE']=points['IRIS_AFFECTE'].map(communes['CODGEO_ETUDE'])
points['CONTROLE_COMMUNE']='NON_RENSEIGNE'
controle=points['IRIS_AFFECTE'].notna() & points['COMMUNE_SOURCE'].notna()
parent_paris=points['CODGEO_SPATIAL_SOURCE'].astype('string').str.fullmatch(r'751\d{2}',na=False) & points['COMMUNE_SOURCE'].eq('75056')
accord=(points['COMMUNE_SOURCE'].eq(points['CODGEO_SPATIAL_SOURCE']) |
        points['COMMUNE_SOURCE'].eq(points['CODGEO_SPATIAL_ETUDE']) | parent_paris).fillna(False)
points.loc[controle,'CONTROLE_COMMUNE']=np.where(accord[controle],'COHERENT','DIFFERENT')
points['INCLUS_COMPTAGE']=points['IRIS_AFFECTE'].notna() & ~points['CONTROLE_COMMUNE'].eq('DIFFERENT')
exporter_csv(points.loc[points['CONTROLE_COMMUNE'].eq('DIFFERENT')].drop(columns='geometry'),
             DOSSIER_INTERIM/'communes_points_incoherentes_j13c.csv')
print('Points exclus pour incohérence de commune :',int(points['CONTROLE_COMMUNE'].eq('DIFFERENT').sum()))


Points exclus pour incohérence de commune : 367


In [68]:
#Compter les observations par IRIS

profil=base.copy()
retenus=points[points['INCLUS_COMPTAGE']].copy()
NOMS_COMPTES={'RESTAURATION':'J13C_NB_OBS_RESTAURATION','TRANSPORTS':'J13C_NB_POINTS_ARRET',
              'EQUIPEMENTS':'J13C_NB_OBS_EQUIPEMENTS'}
DICTIONNAIRE=[]
def decrire(col,libelle,unite,num='',den='',facteur=1):
    d={'COLONNE':col,'LIBELLE':libelle,'UNITE':unite,'NUMERATEUR':num,'DENOMINATEUR':den,
       'FACTEUR':facteur,'PERIMETRE':'Observations retenues après contrôle dans les IRIS de J13b',
       'GEOGRAPHIE_IRIS':GEOGRAPHIE_IRIS}
    DICTIONNAIRE[:]=[x for x in DICTIONNAIRE if x['COLONNE']!=col]
    DICTIONNAIRE.append(d)

for source,col in NOMS_COMPTES.items():
    charge=source in points_par_source
    n=retenus.loc[retenus['SOURCE'].eq(source)].groupby('IRIS_AFFECTE').size()
    profil[col]=profil['IRIS'].map(n).fillna(0).astype('Int64') if charge else pd.Series(pd.NA,index=profil.index,dtype='Int64')
    profil['J13C_SOURCE_'+source+'_CHARGEE']=charge
    decrire(col,SOURCES[source]['unite'],'observations/points',num='Points inclus de '+source)
rapides=retenus[retenus['SOURCE'].eq('RESTAURATION') & retenus['CATEGORIE_CODE'].isin(['RAPIDE','RESTAURATION_RAPIDE'])]
n_rapides=rapides.groupby('IRIS_AFFECTE').size()
col_rapide='J13C_NB_OBS_RESTAURATION_RAPIDE'
profil[col_rapide]=(profil['IRIS'].map(n_rapides).fillna(0).astype('Int64') if 'RESTAURATION' in points_par_source
                    else pd.Series(pd.NA,index=profil.index,dtype='Int64'))
decrire(col_rapide,'Observations géolocalisées de restauration rapide retenues','observations',num='Catégorie RAPIDE ou RESTAURATION_RAPIDE')
surface=iris_polygones.set_index('IRIS').geometry.area/1_000_000
profil['J13C_SURFACE_CONTOUR_KM2']=profil['IRIS'].map(surface)
decrire('J13C_SURFACE_CONTOUR_KM2','Surface du contour IRIS non simplifié','km²',num='Surface EPSG:2154 / 1 000 000')
population=nombre(profil['J13A_POPULATION']).where(lambda s:s.gt(0))
surf=profil['J13C_SURFACE_CONTOUR_KM2'].where(lambda s:s.gt(0))
indicateurs=[('J13C_DENSITE_OBS_RESTAURATION_10000_HAB','J13C_NB_OBS_RESTAURATION',population,10000,
             'Observations géolocalisées de restauration pour 10 000 habitants','observations / 10 000 habitants','J13A_POPULATION'),
            ('J13C_DENSITE_OBS_RAPIDE_10000_HAB',col_rapide,population,10000,
             'Observations géolocalisées de restauration rapide pour 10 000 habitants','observations / 10 000 habitants','J13A_POPULATION'),
            ('J13C_DENSITE_POINTS_ARRET_KM2','J13C_NB_POINTS_ARRET',surf,1,
             'Points d’arrêt de la source par km²','points / km²','J13C_SURFACE_CONTOUR_KM2'),
            ('J13C_PART_RESTAURATION_RAPIDE_PCT',col_rapide,profil['J13C_NB_OBS_RESTAURATION'].where(lambda s:s.gt(0)),100,
             'Part de restauration rapide parmi les observations de restauration retenues','%','J13C_NB_OBS_RESTAURATION')]
for col,num,den,facteur,lib,unite,nom_den in indicateurs:
    profil[col]=(pd.to_numeric(profil[num],errors='coerce')/den*facteur).astype('Float64')
    decrire(col,lib,unite,num,nom_den,facteur)

categories=retenus.groupby(['IRIS_AFFECTE','SOURCE','CATEGORIE_CODE']).size().reset_index(name='NB_OBSERVATIONS')
categories=categories.rename(columns={'IRIS_AFFECTE':'IRIS','CATEGORIE_CODE':'CATEGORIE'})
if not categories.empty:
    categories['NB_SOURCE_IRIS']=categories.groupby(['IRIS','SOURCE'])['NB_OBSERVATIONS'].transform('sum')
    categories['PART_DANS_SOURCE_IRIS_PCT']=categories['NB_OBSERVATIONS']/categories['NB_SOURCE_IRIS']*100
    categories=categories.merge(profil[['IRIS','NOM_IRIS','NOM_COMMUNE_ETUDE']],on='IRIS',how='left',validate='many_to_one')
else:
    for c in ['NB_SOURCE_IRIS','PART_DANS_SOURCE_IRIS_PCT','NOM_IRIS','NOM_COMMUNE_ETUDE']:
        categories[c]=pd.Series(dtype='float64' if c.startswith(('NB_','PART_')) else 'string')
display(profil[['IRIS','NOM_IRIS']+list(NOMS_COMPTES.values())+[col_rapide]].head(10))


,IRIS,NOM_IRIS,J13C_NB_OBS_RESTAURATION,J13C_NB_POINTS_ARRET,J13C_NB_OBS_EQUIPEMENTS,J13C_NB_OBS_RESTAURATION_RAPIDE
0,751031101,Les Archives 1,16,3,104,7
1,751051702,Saint-Victor 2,22,1,118,13
2,751062309,Notre-Dame des Champs 9,25,7,196,10
3,751072503,Saint-Thomas d'Aquin 3,19,2,82,9
4,751135029,Gare 29,2,4,76,0
5,751135104,Maison Blanche 4,25,3,64,6
6,751135206,Croulebarbe 6,10,3,261,2
7,751145512,Petit Montrouge 12,12,3,93,7
8,751187008,Clignancourt 8,25,3,34,7
9,751187017,Clignancourt 17,27,6,208,14


In [69]:
#Vérifier la conservation des observations et la couverture

couverture=[]
for source,p in points.groupby('SOURCE'):
    nb=len(p);compte=int(p['INCLUS_COMPTAGE'].sum());col=NOMS_COMPTES[source]
    assert int(profil[col].sum())==compte
    assert compte+int((~p['INCLUS_COMPTAGE']).sum())==nb
    couverture.append({'SOURCE':source,'NB_POINTS_UNIQUES':nb,'NB_COORDONNEES_VALIDES':int(p['COORDONNEES_VALIDES'].sum()),
        'NB_IRIS_ATTRIBUES':int(p['IRIS_AFFECTE'].notna().sum()),'NB_POINTS_COMPTES':compte,
        'NB_POINTS_NON_COMPTES':nb-compte,'COUVERTURE_COMPTAGE_PCT':compte/nb*100})
couverture=pd.DataFrame(couverture)
assert len(profil)==NOMBRE_IRIS and profil['IRIS'].is_unique and set(profil['IRIS'])==ATTENDUS
assert set(points.loc[points['INCLUS_COMPTAGE'],'IRIS_AFFECTE'])<=ATTENDUS
for col in NOMS_COMPTES.values():assert profil[col].dropna().ge(0).all()
assert profil[col_rapide].le(profil['J13C_NB_OBS_RESTAURATION']).fillna(True).all()
assert profil['J13C_PART_RESTAURATION_RAPIDE_PCT'].dropna().between(0,100).all()
if not categories.empty:
    sommes=categories.groupby(['IRIS','SOURCE'])['PART_DANS_SOURCE_IRIS_PCT'].sum()
    assert np.allclose(sommes,100)
exporter_csv(couverture,DOSSIER_INTERIM/'couverture_affectations_j13c.csv')
bilan_final=points.groupby(['SOURCE','STATUT_AFFECTATION','CONTROLE_COMMUNE','INCLUS_COMPTAGE']).size().reset_index(name='NB_POINTS')
exporter_csv(bilan_final,DOSSIER_INTERIM/'bilan_affectations_finales_j13c.csv')
display(couverture)


,SOURCE,NB_POINTS_UNIQUES,NB_COORDONNEES_VALIDES,NB_IRIS_ATTRIBUES,NB_POINTS_COMPTES,NB_POINTS_NON_COMPTES,COUVERTURE_COMPTAGE_PCT
0,EQUIPEMENTS,467542,446956,446955,446625,20917,95.526177
1,RESTAURATION,49169,47856,47856,47819,1350,97.254368
2,TRANSPORTS,37956,37956,37546,37546,410,98.919802


In [70]:
# Etudier le focus sans supprimer le reste de l'IDF

profil_focus=profil[profil['J13C_DANS_FOCUS']].copy()
iris_focus=set(profil_focus['IRIS'])
cols=['IRIS','NOM_IRIS','NOM_COMMUNE_ETUDE','J13A_POPULATION','J13C_NB_OBS_RESTAURATION',
      'J13C_NB_OBS_RESTAURATION_RAPIDE','J13C_NB_POINTS_ARRET','J13C_DENSITE_OBS_RESTAURATION_10000_HAB']
display(profil_focus[cols].sort_values('J13C_NB_OBS_RESTAURATION',ascending=False).head(25))
exporter_csv(profil_focus,DOSSIER_PROCESSED/'profil_iris_focus_j13c.csv')


,IRIS,NOM_IRIS,NOM_COMMUNE_ETUDE,J13A_POPULATION,J13C_NB_OBS_RESTAURATION,J13C_NB_OBS_RESTAURATION_RAPIDE,J13C_NB_POINTS_ARRET,J13C_DENSITE_OBS_RESTAURATION_10000_HAB
4826,955000201,Centre Ville,Pontoise,4281.31049669453,77,46,33,179.851473
5053,951270104,Préfecture,Cergy,3527.95063150221,74,54,19,209.753502
4873,951270118,Bastide-Horloge,Cergy,3901.27701391836,23,18,17,58.955055
4746,951270106,Port Ham-Village,Cergy,3856.87164536655,21,9,16,54.448273
4953,951270130,Le Point du Jour-Centre Gare,Cergy,11526.8708604748,17,8,27,14.748148
...,...,...,...,...,...,...,...,...
5141,955000901,Centre Hospitalier,Pontoise,1831.3165589778,3,3,8,16.381657
5073,951270115,Les Genottes,Cergy,1780.7768410772,3,2,2,16.84658
4926,951270132,Le Hazay-Les Essarts,Cergy,2431.04726686769,3,2,2,12.340361
5128,951270102,Touleuses-Bocages,Cergy,2508.41626285079,3,1,0,11.959737


In [71]:
#Exporter les CSV et GeoPackages

wgs=points.to_crs(4326)
points['LONGITUDE']=wgs.geometry.x;points['LATITUDE']=wgs.geometry.y
iris_geo=iris_polygones.merge(profil,on='IRIS',how='left',validate='one_to_one')
FICHIER_PROFIL=DOSSIER_PROCESSED/'profil_iris_idf_j13c.csv'
FICHIER_GEO=DOSSIER_PROCESSED/'profil_iris_idf_j13c.gpkg'
FICHIER_POINTS=DOSSIER_PROCESSED/'points_affectations_idf_j13c.csv'
FICHIER_POINTS_GEO=DOSSIER_PROCESSED/'points_geolocalises_j13c.gpkg'
FICHIER_CATEGORIES=DOSSIER_PROCESSED/'categories_points_par_iris_j13c.csv'
FICHIER_DICTIONNAIRE=DOSSIER_PROCESSED/'dictionnaire_indicateurs_j13c.csv'
exporter_csv(profil,FICHIER_PROFIL)
exporter_csv(points.drop(columns='geometry'),FICHIER_POINTS)
exporter_csv(categories,FICHIER_CATEGORIES)
table_dictionnaire=pd.DataFrame(DICTIONNAIRE)
assert table_dictionnaire['COLONNE'].is_unique
exporter_csv(table_dictionnaire,FICHIER_DICTIONNAIRE)

def ecrire_geopackage(table,path,layer):
    temporaire=path.with_name(path.stem+'_temporaire.gpkg')
    if temporaire.exists():temporaire.unlink()
    table.to_file(temporaire,layer=layer,driver='GPKG',index=False)
    temporaire.replace(path)

ecrire_geopackage(iris_geo,FICHIER_GEO,'profils_iris')
valides=points[points['COORDONNEES_VALIDES']].copy()
if not valides.empty:ecrire_geopackage(valides,FICHIER_POINTS_GEO,'points')
elif FICHIER_POINTS_GEO.exists():FICHIER_POINTS_GEO.unlink()  # Retirer une ancienne sortie devenue non pertinente.
print('Profil régional :',FICHIER_PROFIL)
print('Points avec statuts :',FICHIER_POINTS)
print('Catégories par IRIS :',FICHIER_CATEGORIES)


Profil régional : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13c\profil_iris_idf_j13c.csv
Points avec statuts : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13c\points_affectations_idf_j13c.csv
Catégories par IRIS : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13c\categories_points_par_iris_j13c.csv


In [72]:
#Carte régionale des observations de restauration

FICHIERS_HTML=[]
carte_geo=iris_geo.copy()
carte_geo.geometry=carte_geo.geometry.simplify(SIMPLIFICATION_CARTE_METRES,preserve_topology=True)
carte_geo=carte_geo.to_crs(4326)
geoj=json.loads(carte_geo[['IRIS','geometry']].to_json())
x0,y0,x1,y1=carte_geo.total_bounds
def fmt(v):return 'Non disponible' if pd.isna(v) else f'{float(v):,.1f}'.replace(',',' ').replace('.',',')
col='J13C_NB_OBS_RESTAURATION'
hover=['<b>'+escape(str(r['NOM_IRIS']))+'</b><br>'+escape(str(r['NOM_COMMUNE_ETUDE']))
       +'<br>IRIS : '+escape(str(r['IRIS']))+'<br>Population : '+fmt(r['J13A_POPULATION'])
       +'<br>Observations de restauration retenues : '+fmt(r[col])
       +'<br>Pour 10 000 habitants : '+fmt(r['J13C_DENSITE_OBS_RESTAURATION_10000_HAB'])
       for _,r in carte_geo.iterrows()]
fig=go.Figure(go.Choropleth(geojson=geoj,featureidkey='properties.IRIS',locations=carte_geo['IRIS'],
    z=[0]*len(carte_geo),colorscale=[[0,'#dedede'],[1,'#dedede']],showscale=False,
    marker_line_color='white',marker_line_width=.3,text=hover,hovertemplate='%{text}<extra></extra>'))
connus=carte_geo[col].notna()
if connus.any():
    fig.add_trace(go.Choropleth(geojson=geoj,featureidkey='properties.IRIS',locations=carte_geo.loc[connus,'IRIS'],
        z=carte_geo.loc[connus,col].astype(float),colorscale='YlOrRd',colorbar_title='Observations',
        marker_line_color='white',marker_line_width=.3,text=[h for h,b in zip(hover,connus) if b],
        hovertemplate='%{text}<extra></extra>'))
fig.update_geos(visible=False,projection_type='mercator',lonaxis_range=[x0-.04,x1+.04],lataxis_range=[y0-.03,y1+.03])
fig.update_layout(title='Observations de restauration affectées aux IRIS — IDF',height=750,margin=dict(l=10,r=10,t=65,b=10))
f=DOSSIER_CARTES/'01_restauration_par_iris_idf.html'
fig.write_html(str(f),include_plotlyjs=True,full_html=True,config=CONFIG_PLOTLY);FICHIERS_HTML.append(f)
if AFFICHER_CARTES_DANS_NOTEBOOK:fig.show(config=CONFIG_PLOTLY)
print('Carte régionale :',f)


Carte régionale : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\cartes\j13c\01_restauration_par_iris_idf.html


In [73]:
#Carte locale : restaurants, équipements

local=carte_geo[carte_geo['IRIS'].isin(iris_focus)].copy()
if not local.empty:
    geo_local=json.loads(local[['IRIS','geometry']].to_json())
    bx0,by0,bx1,by1=local.total_bounds
    points_locaux=points[points['COORDONNEES_VALIDES'] & points['LONGITUDE'].between(bx0-.01,bx1+.01)
                         & points['LATITUDE'].between(by0-.01,by1+.01)].copy()
    carte=go.Figure(go.Choropleth(geojson=geo_local,featureidkey='properties.IRIS',locations=local['IRIS'],
        z=[0]*len(local),showscale=False,colorscale=[[0,'#f0f3f7'],[1,'#f0f3f7']],
        marker_line_color='#8a94a5',marker_line_width=.8,text=local['NOM_IRIS'],hovertemplate='%{text}<extra></extra>'))
    lib_statuts={'INTERIEUR_UNIQUE':'À l’intérieur d’un seul IRIS',
        'LIMITE_UNIQUE':'Sur une limite : un seul IRIS candidat',
        'LIMITE_MULTIPLE':'À vérifier : limite commune à plusieurs IRIS',
        'INTERIEUR_MULTIPLE':'À vérifier : plusieurs polygones contiennent le point',
        'LIMITE_OU_AMBIGUITE_VALIDEE':'Choix de limite documenté et validé',
        'HORS_IRIS_IDF':'Hors des polygones IRIS du périmètre',
        'COORDONNEES_ABSENTES_OU_INVALIDES':'Coordonnées absentes ou invalides'}
    styles={'RESTAURATION':('Restaurants','#e76f51',7),'TRANSPORTS':('Points d’arrêt','#277da8',6),
            'EQUIPEMENTS':('Équipements complémentaires','#43aa8b',5)}
    for source,(lib,couleur,taille) in styles.items():
        p=points_locaux[points_locaux['SOURCE'].eq(source)]
        if p.empty:continue
        textes=['<b>'+escape('Nom non renseigné dans la source' if pd.isna(r['NOM']) else str(r['NOM']))+'</b>'
            +'<br>Identifiant : '+escape(str(r['ID_SOURCE']))+'<br>Catégorie : '+escape(str(r['CATEGORIE']))
            +'<br>IRIS : '+escape('Non affecté' if pd.isna(r['IRIS_AFFECTE']) else str(r['IRIS_AFFECTE']))
            +'<br>Affectation : '+escape(lib_statuts.get(str(r['STATUT_AFFECTATION']),str(r['STATUT_AFFECTATION'])))
            +'<br>Retenu dans les comptes : '+('Oui' if r['INCLUS_COMPTAGE'] else 'Non')
            for _,r in p.iterrows()]
        carte.add_trace(go.Scattergeo(lon=p['LONGITUDE'],lat=p['LATITUDE'],mode='markers',name=lib,
            marker=dict(size=taille,color=couleur,opacity=.85),text=textes,hovertemplate='%{text}<extra></extra>'))
    carte.update_geos(visible=False,projection_type='mercator',lonaxis_range=[bx0-.01,bx1+.01],lataxis_range=[by0-.01,by1+.01])
    carte.update_layout(title='IRIS et points du secteur du focus — affectations contrôlées',height=750,
                        margin=dict(l=10,r=10,t=65,b=10),legend=dict(orientation='h'))
    f=DOSSIER_CARTES/'02_points_et_iris_focus.html'
    carte.write_html(str(f),include_plotlyjs=True,full_html=True,config=CONFIG_PLOTLY);FICHIERS_HTML.append(f)
    if AFFICHER_CARTES_DANS_NOTEBOOK:carte.show(config=CONFIG_PLOTLY)
    print('Carte locale :',f)
else:print('Focus vide : aucune carte locale produite. Renseigner CODES_FOCUS si nécessaire.')


Carte locale : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\cartes\j13c\02_points_et_iris_focus.html


In [74]:
# Enregistrer la provenance et les limites

sources=[FICHIER_PROFIL_J13B,FICHIER_GEO_J13B]+[c['fichier'] for c in SOURCES.values() if c['actif']]
sources += [Path(p['FICHIER_ORIGINAL']) for p in PREPARATIONS_EFFECTUEES if 'FICHIER_ORIGINAL' in p]
if FICHIER_NOMENCLATURE_BPE is not None:sources.append(Path(FICHIER_NOMENCLATURE_BPE))
if FICHIER_CORRECTIONS_LIMITES is not None:sources.append(Path(FICHIER_CORRECTIONS_LIMITES))
metadata={'etape':'J13C_AFFECTATIONS_POINTS_IRIS_IDF','execution_utc':datetime.now(timezone.utc).isoformat(),
    'annee_rp':ANNEE_RP,'geographie_iris':GEOGRAPHIE_IRIS,'crs_travail':CRS_TRAVAIL,
    'nb_iris':len(profil),'nb_iris_focus':len(profil_focus),'codes_focus':sorted(focus_codes),
    'nb_points':len(points),'nb_points_comptes':int(points['INCLUS_COMPTAGE'].sum()),
    'nb_corrections_limites':len(corrections_utilisees),'lectures':LECTURES,
    'sources':[{'fichier':str(p),'sha256':empreinte(p)} for p in dict.fromkeys(sources)],
    'qualite_sources':statistiques_sources.to_dict(orient='records'),
    'provenance_sources_declarees':{nom:{k:cfg.get(k) for k in ['annee_source','date_source','geographie_source','url_notice','unite']} for nom,cfg in SOURCES.items()},
    'preparations_sources':PREPARATIONS_EFFECTUEES,'etat_entrees':etat.to_dict(orient='records'),
    'couverture_affectations':couverture.to_dict(orient='records'),
    'predicats':['within','intersects'],'cartes':[str(p) for p in FICHIERS_HTML],
    'logiciels':{'pandas':pd.__version__,'geopandas':gpd.__version__,'plotly':plotly.__version__},
    'limites':['Comptes d’observations retenues de la source, pas inventaire garanti exhaustif.',
        'Clés de ligne possibles pour la BPE : elles ne certifient pas l’unicité économique.',
        'Points d’arrêt selon l’identifiant choisi, pas automatiquement gares commerciales.',
        'Points ambigus non comptés sans correction documentée ; aucune affectation au plus proche.',
        'Incohérences commune/position exclues du comptage.',
        'Millésime des points distinct du millésime des profils et de la géographie IRIS.',
        'TRANSPORTS décrit seulement le fichier sélectionné, selon son grain : arrêt, zone ou observation par ligne.',
        'Les clés de ligne BPE sont liées à une version du fichier, pas des identifiants économiques permanents.',
        'Localisation, pas fréquentation, fréquence de service, temps de marche ou flux.',
        'Zéro correspond à aucun point retenu ; source non chargée et dénominateur nul donnent NA.',
        'Les revenus et leurs statuts sont conservés depuis J13b, sans réestimation.']}
FICHIER_METADATA=DOSSIER_PROCESSED/'metadata_affectations_iris_j13c.json'
FICHIER_METADATA.write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('Métadonnées :',FICHIER_METADATA)


registre=[]
for p in metadata['sources']:
    chemin=Path(p['fichier']);roles=[];familles=set()
    for source,cfg in SOURCES.items():
        if cfg['actif'] and Path(cfg['fichier'])==chemin:
            roles.append('ENTREE_'+source);familles.add(source)
    for preparation in PREPARATIONS_EFFECTUEES:
        concerne=any(preparation.get(k) and Path(preparation[k])==chemin for k in ['FICHIER_ORIGINAL','SORTIE'])
        if not concerne:continue
        role=preparation['ETAPE']
        if preparation.get('SOURCE'):
            role+='_'+preparation['SOURCE'];familles.add(preparation['SOURCE'])
        if preparation['ETAPE']=='EXTRACTION_BPE_IDF':familles.add('BPE')
        roles.append(role)
    if FICHIER_BPE_DETAIL is not None and chemin==Path(FICHIER_BPE_DETAIL):
        roles.append('BPE_DETAIL');familles.add('BPE')
    if FICHIER_ARRETS_IDFM is not None and chemin==Path(FICHIER_ARRETS_IDFM):
        roles.append('IDFM_ORIGINAL');familles.add('TRANSPORTS')
    if chemin in [FICHIER_PROFIL_J13B,FICHIER_GEO_J13B]:roles=['PROFIL_IRIS_J13B']
    if FICHIER_NOMENCLATURE_BPE is not None and chemin==Path(FICHIER_NOMENCLATURE_BPE):
        roles=['NOMENCLATURE_BPE'];familles.add('BPE')
    bpe=bool(familles & {'BPE','RESTAURATION','EQUIPEMENTS'})
    idfm='TRANSPORTS' in familles
    j13b='PROFIL_IRIS_J13B' in roles
    registre.append({'ROLE':', '.join(dict.fromkeys(roles)) or 'SOURCE_DE_PROJET',
        'PRODUCTEUR':'Insee / préparation projet' if bpe else 'Île-de-France Mobilités / préparation projet' if idfm else 'Projet / sources J13b' if j13b else 'Projet',
        'FICHIER':p['fichier'],'URL_NOTICE':URL_NOTICE_BPE if bpe else URL_NOTICE_IDFM if idfm else '',
        'ANNEE_SOURCE':ANNEE_BPE if bpe else ANNEE_SOURCE_IDFM if idfm else '',
        'GEOGRAPHIE_SOURCE':GEOGRAPHIE_BPE if bpe else GEOGRAPHIE_IRIS if j13b else '',
        'DATE_SOURCE':DATE_SOURCE_BPE if bpe else DATE_SOURCE_IDFM if idfm else '',
        'DETAIL_MILLESIMES':('RP '+str(ANNEE_RP)+' ; revenus : voir provenance et statuts J13b') if j13b else '',
        'DATE_EXECUTION_UTC':metadata['execution_utc'],'SHA256':p['sha256']})
FICHIER_DOC_SOURCES=DOSSIER_DOCS/'sources_j13c.csv'
exporter_csv(pd.DataFrame(registre),FICHIER_DOC_SOURCES)
print('Registre des sources :',FICHIER_DOC_SOURCES)


Métadonnées : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\j13c\metadata_affectations_iris_j13c.json
Registre des sources : C:\Users\almou\OneDrive\GeoMarketing_IDF\docs\sources_j13c.csv


In [75]:
#Préparer la première documentation à versionner

sources_doc='\n'.join('- '+r['SOURCE']+' : `'+r['FICHIER']+'` ; unité : '+r['UNITE'] for r in LECTURES)
doc=f"""# J13c — Sources IDFM/BPE et affectations aux IRIS IDF

## But

Affecter les observations de restauration, les points d’arrêt et les équipements
complémentaires aux IRIS du profil J13b. Conserver tous les IRIS et diagnostiquer
les points qui ne peuvent pas être comptés.

## Entrées

- Profil et contours J13b ; RP {ANNEE_RP}, géographie {GEOGRAPHIE_IRIS}.
{sources_doc}

Les points sont des tables de détail. Les arrêts ferrés et les équipements sont
normalisés dans data/processed/j13c/entrees. L’extrait BPE éventuel est lu par blocs
et conservé dans data/interim/j13c. Les tracés ferroviaires restent dans J12.
Les restaurants et équipements peuvent être construits directement depuis le détail BPE.
Les arrêts IDFM sont préparés selon leur schéma et leur unité de comptage.
Le registre docs/sources_j13c.csv contient les notices déclarées, dates et empreintes.
Les équipements utilisent les libellés documentés de J10 ou de la nomenclature
BPE lorsque celle-ci est fournie. Les codes sans libellé restent à documenter.
Les préparations, leurs fichiers originaux et leurs unités figurent dans les métadonnées.

Les points sont des tables de détail. Les dates des sources et identifiants choisis
figurent dans les métadonnées locales J13c. Les fichiers raw restent intacts.

## Méthode

Dédupliquer selon la clé de la source ; refuser les identifiants aux positions ou
catégories divergentes. Travailler en EPSG:{CRS_TRAVAIL}. Affecter par within, puis
intersects. Conserver les limites multiples sans affectation automatique. Accepter
uniquement une correction documentée parmi les candidats. Exclure du comptage les
points dont la commune source contredit les communes source/étude de J13b.

## Sorties

CSV : profil régional, focus, points et statuts, catégories et dictionnaire.
GeoPackages : contours avec indicateurs et points géolocalisés.
HTML : carte régionale et carte locale du focus, sans API key ni OpenStreetMap.
Diagnostics : coordonnées, doublons, candidats, points non affectés et couverture.

## Contrôles

IRIS uniques et périmètre conservé ; clé de point unique ; comptes IRIS égaux aux
points retenus ; bilan des points comptés/non comptés ; pourcentages entre 0 et 100 ;
dictionnaire unique ; fichiers de sortie existants.

## Limites

Les comptes décrivent les observations de la source. Zéro ne prouve pas l’absence
réelle. Une source désactivée donne NA. Les unités ne sont pas additionnées entre
sources. Les points d’arrêt ne sont pas automatiquement des gares commerciales.
Les comptes par IRIS ne mesurent ni flux, ni fréquentation, ni temps de marche,
ni zone de chalandise. Les revenus et leur méthode restent ceux de J13b.

## Suite

J13d : mobilités professionnelles et scolaires entre communes, en gardant les
échanges hors IDF. J13e : lecture locale croisée. J14 : comparaison de sites.

## Références techniques

- https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html
- https://docs.github.com/en/repositories/working-with-files/managing-files/adding-a-file-to-a-repository
- https://nbconvert.readthedocs.io/en/latest/config_options.html
"""
FICHIER_DOC=DOSSIER_DOCS/'j13c.md'
FICHIER_DOC.write_text(doc,encoding='utf-8')
exporter_csv(table_dictionnaire,DOSSIER_DOCS/'dictionnaire_j13c.csv')
print('Documentation prête :',FICHIER_DOC)


fichiers_doc=[{'ROLE':'ENTREE_J13B','FICHIER':str(FICHIER_PROFIL_J13B.relative_to(RACINE)),
              'USAGE':'Profil régional à conserver'},
             {'ROLE':'CONTOURS_J13B','FICHIER':str(FICHIER_GEO_J13B.relative_to(RACINE)),
              'USAGE':'Contours complets pour l’affectation'}]
for source,cfg in SOURCES.items():
    if cfg['actif']:
        fichiers_doc.append({'ROLE':'ENTREE_'+source,'FICHIER':str(cfg['fichier']),
            'USAGE':cfg['unite']})
for p in PREPARATIONS_EFFECTUEES:
    fichiers_doc.append({'ROLE':'SOURCE_ORIGINALE','FICHIER':p['FICHIER_ORIGINAL'],
        'USAGE':p['ETAPE']})
for path in [FICHIER_PROFIL,FICHIER_GEO,FICHIER_POINTS,FICHIER_CATEGORIES,FICHIER_DICTIONNAIRE,FICHIER_METADATA]:
    fichiers_doc.append({'ROLE':'SORTIE','FICHIER':str(path.relative_to(RACINE)),
        'USAGE':'Résultat J13c'})
for path in FICHIERS_HTML:
    fichiers_doc.append({'ROLE':'CARTE_HTML','FICHIER':str(path.relative_to(RACINE)),
        'USAGE':'Carte à ouvrir dans le navigateur ; noms au survol'})
FICHIER_DOC_FICHIERS=DOSSIER_DOCS/'fichiers_j13c.csv'
exporter_csv(pd.DataFrame(fichiers_doc).drop_duplicates(),FICHIER_DOC_FICHIERS)
print('Liste des fichiers :',FICHIER_DOC_FICHIERS)


Documentation prête : C:\Users\almou\OneDrive\GeoMarketing_IDF\docs\j13c.md
Liste des fichiers : C:\Users\almou\OneDrive\GeoMarketing_IDF\docs\fichiers_j13c.csv


## Conclusion

| Observation | Formulation possible | Ce qu’il reste à examiner |
|---|---|---|
| Volume élevé de restaurants géolocalisés | L’offre de restauration de la source se concentre dans ce quartier | Type d’offre, horaires, prix, concurrence au-delà de la limite |
| Beaucoup de points d’arrêt | Plusieurs points du réseau sont localisés dans le quartier | Unité de comptage, modes, desserte, horaires et accès piéton |
| Beaucoup de ménages d’une personne dans J13b | Public résident pouvant justifier une offre de repas individuels | Habitudes, budget et lieux réellement fréquentés |
| Forte densité et faible population | Quartier pouvant avoir une fonction d’activité ou de transit | Emplois présents, commerces, campus et fréquentation |
| Peu de points et couverture incomplète | Peu d’observations affectées avec la source disponible | Coordonnées, complétude et contrôle terrain |

Exemple : « Le quartier compte X observations de restauration géolocalisées retenues,
dont Y de restauration rapide. Cela représente Z observations pour 10 000 résidents.
Ces résultats décrivent l’offre localisée de la source et doivent être confrontés
aux flux, aux prix, aux horaires et aux établissements voisins. »